In [1]:
%load_ext cython

In [74]:
%load_ext cython
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, Any, Optional, List, Tuple
import warnings
import json
import time
from pathlib import Path
import threading
import queue
import concurrent.futures
import requests
import hashlib
import pickle
import os
from dataclasses import dataclass, asdict
from abc import ABC, abstractmethod
import re
from scipy.spatial.distance import euclidean
from scipy.stats import pearsonr

warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("=== ENHANCED LLM-INSPIRED MOACP FRAMEWORK WITH ADAPTIVE OPERATORS ===")

The cython extension is already loaded. To reload it, use:
  %reload_ext cython
=== ENHANCED LLM-INSPIRED MOACP FRAMEWORK WITH ADAPTIVE OPERATORS ===


In [75]:
class AdaptiveOperatorManager:
    """Advanced operator management inspired by hyper-heuristics literature"""
    
    def __init__(self):
        # Extended operator set with specialized functions
        self.available_operators = {
            'swap': {
                'description': 'Exchange items in solution',
                'complexity': 'low',
                'best_for': ['small', 'medium'],
                'objectives': [2, 3, 4],
                'exploration': 0.6,
                'exploitation': 0.4,
                'reliability': 0.9  # Added reliability score
            },
            'greedy_add': {
                'description': 'Add best feasible items',
                'complexity': 'medium',
                'best_for': ['medium', 'large'],
                'objectives': [2, 3, 4],
                'exploration': 0.3,
                'exploitation': 0.7,
                'reliability': 0.85
            },
            'mutation': {
                'description': 'Random modifications',
                'complexity': 'low',
                'best_for': ['small', 'medium'],
                'objectives': [2, 3, 4],
                'exploration': 0.8,
                'exploitation': 0.2,
                'reliability': 0.8
            },
            'local_search': {
                'description': 'Neighborhood exploration',
                'complexity': 'high',
                'best_for': ['medium', 'large'],
                'objectives': [2, 3, 4],
                'exploration': 0.2,
                'exploitation': 0.8,
                'reliability': 0.95
            },
            'repair': {
                'description': 'Fix infeasible solutions',
                'complexity': 'medium',
                'best_for': ['very_tight'],
                'objectives': [2, 3, 4],
                'exploration': 0.4,
                'exploitation': 0.6,
                'reliability': 0.9
            },
            'recombine': {
                'description': 'Combine solution parts',
                'complexity': 'medium',
                'best_for': ['small', 'medium'],
                'objectives': [3, 4],
                'exploration': 0.7,
                'exploitation': 0.3,
                'reliability': 0.75
            },
            'intensify': {
                'description': 'Intensify search around good solutions',
                'complexity': 'high',
                'best_for': ['large'],
                'objectives': [2],
                'exploration': 0.1,
                'exploitation': 0.9,
                'reliability': 0.85
            },
            'diversify': {
                'description': 'Explore diverse regions',
                'complexity': 'medium',
                'best_for': ['small', 'medium'],
                'objectives': [3, 4],
                'exploration': 0.9,
                'exploitation': 0.1,
                'reliability': 0.7
            }
        }
        
        self.operator_performance = {}
        self.usage_history = {}
        
    def get_optimal_operators(self, instance_features, generation=0):
        """Enhanced operator selection with better performance tracking"""
        
        # Determine optimal number of operators
        num_operators = self._get_optimal_operator_count(instance_features)
        
        # Filter operators suitable for this instance
        suitable_operators = []
        for op_name, op_info in self.available_operators.items():
            if (instance_features['instance_size'] in op_info['best_for'] or
                instance_features.get('capacity_tightness') == 'very_tight' and op_name == 'repair' or
                instance_features['n_objectives'] in op_info['objectives']):
                suitable_operators.append(op_name)
        
        # If no suitable operators found, use defaults based on instance size
        if not suitable_operators:
            if instance_features['instance_size'] == 'small':
                suitable_operators = ['swap', 'greedy_add', 'local_search', 'mutation']
            elif instance_features['instance_size'] == 'medium':
                suitable_operators = ['swap', 'greedy_add', 'local_search']
            else:  # large
                suitable_operators = ['greedy_add', 'local_search', 'intensify']
        
        # Calculate exploration/exploitation balance based on generation
        exploration_rate = max(0.2, 0.8 - (generation * 0.08))
        
        # Score operators based on historical performance
        operator_scores = []
        for op in suitable_operators:
            op_info = self.available_operators[op]
            
            # Base score from historical performance
            base_score = self.operator_performance.get(op, {}).get('success_rate', 0.5)
            
            # Adjust based on exploration/exploitation needs
            if exploration_rate > 0.5:
                score = base_score * 0.7 + op_info['exploration'] * 0.3
            else:
                score = base_score * 0.7 + op_info['exploitation'] * 0.3
            
            # Boost operators that have historically performed well
            if op in ['swap', 'greedy_add', 'local_search']:
                score *= 1.1  # These are generally reliable operators
            
            # Adjust for instance-specific factors
            if instance_features.get('capacity_tightness') == 'very_tight' and op == 'repair':
                score *= 1.2
            if instance_features['n_objectives'] > 2 and op in ['diversify', 'recombine']:
                score *= 1.1
            if instance_features['instance_size'] == 'large' and op in ['intensify', 'greedy_add']:
                score *= 1.1
            
            # Factor in reliability
            score *= op_info.get('reliability', 0.8)
            
            operator_scores.append((op, score))
        
        # Sort by score and select top operators
        operator_scores.sort(key=lambda x: x[1], reverse=True)
        selected_operators = [op for op, _ in operator_scores[:num_operators]]
        
        return selected_operators
    
    def _get_optimal_operator_count(self, instance_features):
        """Determine optimal number of operators based on instance characteristics"""
        
        if instance_features['instance_size'] == 'small':
            if instance_features['n_objectives'] <= 2:
                return 4
            else:
                return 5
        elif instance_features['instance_size'] == 'medium':
            if instance_features['n_objectives'] <= 2:
                return 3
            else:
                return 4
        else:  # large
            if instance_features['n_objectives'] <= 2:
                return 2
            else:
                return 3
    
    def update_operator_performance(self, operator, improvement, runtime):
        """Track operator performance for future selection"""
        if operator not in self.operator_performance:
            self.operator_performance[operator] = {
                'improvements': [],
                'runtimes': [],
                'success_rate': 0.0,
                'avg_runtime': 0.0,
                'usage_count': 0
            }
        
        self.operator_performance[operator]['improvements'].append(improvement)
        self.operator_performance[operator]['runtimes'].append(runtime)
        self.operator_performance[operator]['usage_count'] += 1
        
        # Update statistics
        improvements = self.operator_performance[operator]['improvements']
        self.operator_performance[operator]['success_rate'] = sum(1 for imp in improvements if imp > 0) / len(improvements)
        self.operator_performance[operator]['avg_runtime'] = np.mean(self.operator_performance[operator]['runtimes'])
    
    def get_operator_statistics(self):
        """Get current operator performance statistics"""
        stats = {}
        for op, perf in self.operator_performance.items():
            stats[op] = {
                'success_rate': perf['success_rate'],
                'avg_runtime': perf['avg_runtime'],
                'usage_count': perf['usage_count'],
                'avg_improvement': np.mean(perf['improvements']) if perf['improvements'] else 0.0
            }
        return stats

# Initialize adaptive operator manager
operator_manager = AdaptiveOperatorManager()

In [76]:
class InstanceSpecificConfigManager:
    """Enhanced configuration manager with adaptive operator selection"""
    
    def __init__(self):
        self.instance_features = {}
        self.config_registry = {}
        self.performance_history = {}
        self.best_configs = {}
        self.subclass_configs = {}
        self.instance_similarity_matrix = {}
        self.operator_manager = AdaptiveOperatorManager()
        
    def extract_instance_features(self, instance_file, weights_file, nbitems, num_objectives):
        """Enhanced feature extraction with better error handling"""
        features = {
            'n_items': nbitems,
            'n_objectives': num_objectives,
            'instance_size': 'small' if nbitems <= 250 else 'medium' if nbitems <= 500 else 'large',
            'complexity': num_objectives * nbitems / 1000
        }
        
        try:
            # Extract more detailed features from the instance file
            with open(instance_file, 'r') as f:
                lines = f.readlines()
                
                # Try different parsing strategies
                if len(lines) > 0:
                    first_line = lines[0].strip()
                    if ':' in first_line:  # Handle format like "1: 2 250"
                        parts = first_line.split(':')
                        if len(parts) >= 2:
                            nums = parts[1].strip().split()
                            if len(nums) >= 2:
                                features['n_objectives'] = int(nums[0])
                                features['n_items'] = int(nums[1])
                    else:  # Handle space-separated format
                        parts = first_line.split()
                        if len(parts) >= 2:
                            features['n_objectives'] = int(parts[0])
                            features['n_items'] = int(parts[1])
            
            # Calculate capacity distribution features
            capacities = []
            # Find capacity lines (skip first line which contains dimensions)
            capacity_line_idx = 1
            if ':' in lines[0]:  # If first line has format "1: 2 250"
                capacity_line_idx = 1
            else:  # If first line has format "2 250"
                capacity_line_idx = 1
            
            for i in range(capacity_line_idx, min(capacity_line_idx + features['n_objectives'], len(lines))):
                if lines[i].strip():
                    try:
                        capacities.append(float(lines[i].strip()))
                    except ValueError:
                        # Skip lines that can't be parsed as float
                        continue
            
            if capacities:
                features['capacity_mean'] = np.mean(capacities)
                features['capacity_std'] = np.std(capacities)
                features['capacity_ratio'] = features['capacity_mean'] / nbitems
                
                # More granular capacity classification
                if features['capacity_ratio'] > 0.8:
                    features['capacity_tightness'] = 'very_tight'
                elif features['capacity_ratio'] > 0.6:
                    features['capacity_tightness'] = 'tight'
                elif features['capacity_ratio'] > 0.4:
                    features['capacity_tightness'] = 'medium'
                else:
                    features['capacity_tightness'] = 'loose'
            else:
                # Default values if no capacities found
                features['capacity_mean'] = nbitems * 0.5
                features['capacity_std'] = nbitems * 0.1
                features['capacity_ratio'] = 0.5
                features['capacity_tightness'] = 'medium'
            
            # Extract item weight/profit correlation features
            weights = []
            profits = []
            # Find where item data starts
            item_start_idx = capacity_line_idx + features['n_objectives']
            
            # Parse item data (3 lines per item: index, weight, profit)
            for i in range(item_start_idx, len(lines), 3):
                try:
                    if i + 2 < len(lines):
                        weight_line = lines[i + 1].strip()
                        profit_line = lines[i + 2].strip()
                        
                        if weight_line and profit_line:
                            weight_val = float(weight_line)
                            profit_val = float(profit_line)
                            
                            weights.append(weight_val)
                            profits.append(profit_val)
                except (ValueError, IndexError):
                    continue
            
            if weights and profits:
                features['weight_profit_correlation'] = np.corrcoef(weights, profits)[0, 1]
                features['weight_variance'] = np.var(weights)
                features['profit_variance'] = np.var(profits)
                
                # Calculate profit-to-weight ratio distribution
                profit_weight_ratios = [p/w for p, w in zip(profits, weights) if w > 0]
                if profit_weight_ratios:
                    features['pw_ratio_mean'] = np.mean(profit_weight_ratios)
                    features['pw_ratio_std'] = np.std(profit_weight_ratios)
                    features['pw_ratio_skewness'] = self._calculate_skewness(profit_weight_ratios)
                else:
                    features['pw_ratio_mean'] = 1.0
                    features['pw_ratio_std'] = 0.5
                    features['pw_ratio_skewness'] = 0.0
            else:
                # Default values
                features['weight_profit_correlation'] = 0.0
                features['weight_variance'] = 1.0
                features['profit_variance'] = 1.0
                features['pw_ratio_mean'] = 1.0
                features['pw_ratio_std'] = 0.5
                features['pw_ratio_skewness'] = 0.0
        
        except Exception as e:
            print(f"Error extracting features: {e}")
            # Set default values
            features['capacity_ratio'] = 0.5
            features['capacity_tightness'] = 'medium'
            features['weight_profit_correlation'] = 0.0
            features['weight_variance'] = 1.0
            features['profit_variance'] = 1.0
            features['pw_ratio_mean'] = 1.0
            features['pw_ratio_std'] = 0.5
            features['pw_ratio_skewness'] = 0.0
        
        # Create more specific subclass based on multiple features
        features['subclass'] = f"{features['capacity_tightness']}_capacity_{features['n_objectives']}obj"
        
        # Add correlation dimension
        if features['weight_profit_correlation'] > 0.7:
            features['subclass'] += '_high_corr'
        elif features['weight_profit_correlation'] < -0.7:
            features['subclass'] += '_neg_corr'
        else:
            features['subclass'] += '_low_corr'
        
        # Add profit-weight ratio dimension
        if features.get('pw_ratio_skewness', 0) > 1.0:
            features['subclass'] += '_high_skew'
        elif features.get('pw_ratio_skewness', 0) < -1.0:
            features['subclass'] += '_low_skew'
        else:
            features['subclass'] += '_symmetric'
        
        signature = f"{features['n_items']}_{features['n_objectives']}"
        features['signature'] = signature
        
        self.instance_features[signature] = features
        return features
    
    def _calculate_skewness(self, data):
        """Calculate skewness of a dataset"""
        if len(data) < 3:
            return 0.0
        mean = np.mean(data)
        std = np.std(data)
        if std == 0:
            return 0.0
        return np.mean(((data - mean) / std) ** 3)
    
    def get_best_config_for_subclass(self, subclass, generation=0):
        """Get best configuration for a specific subclass with adaptive operators"""
        
        # Get instance features for this subclass
        instance_features = {
            'instance_size': 'medium' if 'medium' in subclass else 'small' if 'small' in subclass else 'large',
            'n_objectives': int(subclass.split('_')[2].replace('obj', '')),
            'capacity_tightness': subclass.split('_')[0]
        }
        
        # Get optimal operators for this subclass and generation
        optimal_operators = self.operator_manager.get_optimal_operators(instance_features, generation)
        
        # Base configurations based on subclass
        base_configs = {
            'very_tight_capacity_2obj': {
                'alpha': 50, 'kappa': 0.18, 'L': 8,
                'runtime_threshold': 12.0, 'search_intensity': 'high'
            },
            'tight_capacity_2obj': {
                'alpha': 45, 'kappa': 0.15, 'L': 7,
                'runtime_threshold': 10.0, 'search_intensity': 'high'
            },
            'medium_capacity_2obj': {
                'alpha': 40, 'kappa': 0.15, 'L': 6,
                'runtime_threshold': 8.0, 'search_intensity': 'medium'
            },
            'loose_capacity_2obj': {
                'alpha': 35, 'kappa': 0.12, 'L': 5,
                'runtime_threshold': 7.0, 'search_intensity': 'medium'
            },
            'tight_capacity_3obj': {
                'alpha': 45, 'kappa': 0.15, 'L': 7,
                'runtime_threshold': 12.0, 'search_intensity': 'high'
            },
            'tight_capacity_4obj': {
                'alpha': 50, 'kappa': 0.18, 'L': 8,
                'runtime_threshold': 15.0, 'search_intensity': 'high'
            }
        }
        
        config = base_configs.get(subclass, base_configs['medium_capacity_2obj']).copy()
        config['operator_strategy'] = optimal_operators
        
        return config
    
    def update_best_config(self, subclass, config, performance, runtime):
        """Update best configuration for a subclass"""
        if subclass not in self.subclass_configs:
            self.subclass_configs[subclass] = config.copy()
            return
        
        # Update if performance is better
        current_best = self.subclass_configs[subclass]
        if performance > self.performance_history.get(subclass, {}).get('best_performance', 0):
            self.subclass_configs[subclass] = config.copy()
            self.performance_history[subclass] = {
                'best_performance': performance,
                'config': config.copy(),
                'timestamp': time.time()
            }
        
        # Update operator performance
        for operator in config.get('operator_strategy', []):
            self.operator_manager.update_operator_performance(operator, performance, runtime)
    
    def calculate_instance_similarity(self, instance1, instance2):
        """Calculate similarity between two instances for knowledge transfer"""
        features1 = self.instance_features.get(instance1, {})
        features2 = self.instance_features.get(instance2, {})
        
        if not features1 or not features2:
            return 0.0
        
        # Calculate similarity for each feature
        similarities = []
        weights = []
        
        # Size similarity
        size_sim = 1 - abs(features1.get('n_items', 0) - features2.get('n_items', 0)) / max(features1.get('n_items', 1), features2.get('n_items', 1))
        similarities.append(size_sim)
        weights.append(0.2)
        
        # Objective count similarity
        obj_sim = 1 if features1.get('n_objectives', 0) == features2.get('n_objectives', 0) else 0
        similarities.append(obj_sim)
        weights.append(0.3)
        
        # Capacity tightness similarity
        capacity_values = {'very_tight': 0.9, 'tight': 0.7, 'medium': 0.5, 'loose': 0.3}
        cap1 = capacity_values.get(features1.get('capacity_tightness', 'medium'), 0.5)
        cap2 = capacity_values.get(features2.get('capacity_tightness', 'medium'), 0.5)
        capacity_sim = 1 - abs(cap1 - cap2)
        similarities.append(capacity_sim)
        weights.append(0.2)
        
        # Correlation similarity
        corr1 = features1.get('weight_profit_correlation', 0)
        corr2 = features2.get('weight_profit_correlation', 0)
        corr_sim = 1 - abs(corr1 - corr2)
        similarities.append(corr_sim)
        weights.append(0.15)
        
        # Profit-weight ratio similarity
        pw1 = features1.get('pw_ratio_mean', 1.0)
        pw2 = features2.get('pw_ratio_mean', 1.0)
        pw_sim = 1 - abs(pw1 - pw2) / max(pw1, pw2)
        similarities.append(pw_sim)
        weights.append(0.15)
        
        # Calculate weighted similarity
        if sum(weights) > 0:
            return sum(s * w for s, w in zip(similarities, weights)) / sum(weights)
        return 0.0
    
    def find_similar_instances(self, target_instance, threshold=0.7):
        """Find instances similar to the target instance for knowledge transfer"""
        similar_instances = []
        for instance in self.instance_features:
            if instance != target_instance:
                similarity = self.calculate_instance_similarity(target_instance, instance)
                if similarity >= threshold:
                    similar_instances.append((instance, similarity))
        
        # Sort by similarity (descending)
        similar_instances.sort(key=lambda x: x[1], reverse=True)
        return similar_instances
    
    def transfer_knowledge(self, target_instance, generation=0):
        """Transfer knowledge from similar instances to the target instance"""
        similar_instances = self.find_similar_instances(target_instance, threshold=0.7)
        
        if not similar_instances:
            return None
        
        # Get the most similar instance
        source_instance, similarity = similar_instances[0]
        
        # Get the best config for the source instance
        source_config = self.best_configs.get(source_instance, {}).get('config')
        
        if not source_config:
            return None
        
        # Adapt the source config for the target instance
        target_features = self.instance_features.get(target_instance, {})
        source_features = self.instance_features.get(source_instance, {})
        
        adapted_config = source_config.copy()
        
        # Adjust parameters based on size difference
        size_ratio = target_features.get('n_items', 1) / source_features.get('n_items', 1)
        adapted_config['alpha'] = int(source_config['alpha'] * size_ratio)
        adapted_config['alpha'] = min(60, max(20, adapted_config['alpha']))
        
        adapted_config['L'] = int(source_config['L'] * size_ratio)
        adapted_config['L'] = min(12, max(2, adapted_config['L']))
        
        # Adjust runtime threshold based on instance size
        if target_features.get('instance_size') == 'small':
            adapted_config['runtime_threshold'] = min(10.0, source_config['runtime_threshold'] * 0.8)
        elif target_features.get('instance_size') == 'large':
            adapted_config['runtime_threshold'] = max(15.0, source_config['runtime_threshold'] * 1.2)
        
        # Update operators for current generation
        optimal_operators = self.operator_manager.get_optimal_operators(target_features, generation)
        adapted_config['operator_strategy'] = optimal_operators
        
        adapted_config['reasoning'] = f"Transferred from {source_instance} (similarity: {similarity:.2f})"
        adapted_config['transferred'] = True
        
        return adapted_config

# Initialize instance-specific config manager
instance_config_manager = InstanceSpecificConfigManager()

In [85]:
class MindEvolutionLLMInterface:
    """Enhanced LLM interface with adaptive operator selection"""
    
    def __init__(self, model_path="llama3:latest", temperature=0.7, max_tokens=500):
        self.model_path = model_path
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.cache = {}
        self.call_count = 0
        self.successful_calls = 0
        self.failed_calls = 0
        self.connection_status = None
        self.exploration_history = []
        self.population_configs = []
        self.knowledge_base = {}
        self.operator_manager = AdaptiveOperatorManager()
        self._verify_connection()
        
    def _verify_connection(self):
        """Verify if LLaMA-3 is accessible"""
        try:
            response = requests.get('http://localhost:11434/api/tags', timeout=5)
            if response.status_code == 200:
                models = response.json().get('models', [])
                model_names = [model['name'] for model in models]
                
                if self.model_path in model_names:
                    self.connection_status = "ollama_connected"
                    print(f"✅ Ollama connected with {self.model_path} available")
                    return True
                else:
                    self.connection_status = "ollama_no_model"
                    return False
            else:
                self.connection_status = "ollama_failed"
        except Exception as e:
            self.connection_status = "ollama_unavailable"
        
        return False
    
    def _call_ollama(self, prompt):
        """Call local Ollama API for LLaMA-3"""
        try:
            response = requests.post(
                'http://localhost:11434/api/generate',
                json={
                    'model': self.model_path,
                    'prompt': prompt,
                    'stream': False,
                    'options': {
                        'temperature': self.temperature,
                        'num_predict': self.max_tokens,
                        'timeout': 30
                    }
                },
                timeout=120
            )
            if response.status_code == 200:
                self.successful_calls += 1
                return response.json()['response']
            else:
                self.failed_calls += 1
                return None
        except Exception as e:
            self.failed_calls += 1
            return None
    

#####
    def _extract_json_from_response(self, response):
        """Simplified but robust JSON extraction"""
        if not response:
            return None
        
        # Find JSON content between braces
        start_idx = response.find('{')
        end_idx = response.rfind('}') + 1
        
        if start_idx == -1 or end_idx == 0:
            return None
        
        json_str = response[start_idx:end_idx]
        
        # Simple fixes for common issues
        json_str = json_str.replace('\n', ' ').replace('\r', '').replace('\t', ' ')
        json_str = re.sub(r'"\s*"', '","', json_str)  # Fix missing commas
        json_str = re.sub(r',\s*}', '}', json_str)   # Fix trailing commas
        
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            # Manual extraction as fallback
            return self._extract_values_manually(json_str)
    def _fix_json_issues_comprehensive(self, json_str):
        """Comprehensive JSON fixing for all types of formatting issues"""
        # Fix broken string concatenations (like "medium""reasoning")
        json_str = re.sub(r'"\s*"\s*', '', json_str)
        
        # Fix missing commas between objects
        json_str = re.sub(r'}\s*{', '},{', json_str)
        json_str = re.sub(r'"\s*}\s*"', '","', json_str)
        json_str = re.sub(r'"\s*\]\s*"', '","', json_str)
        
        # Fix missing commas in arrays
        json_str = re.sub(r'"\s*"\s*', '","', json_str)
        
        # Fix trailing commas
        json_str = re.sub(r',\s*}', '}', json_str)
        json_str = re.sub(r',\s*]', ']', json_str)
        
        # Fix unescaped quotes in values
        json_str = re.sub(r':\s*"([^"]*)"([^"]*?)"', r': "\1\\"\\2\\""', json_str)
        
        # Fix broken key-value pairs
        json_str = re.sub(r'(\w+)\s*:\s*"', r'"\1": "', json_str)
        
        # Fix common LLM output issues
        json_str = re.sub(r':\s*"', ':"', json_str)  # Fix spacing before quotes
        json_str = re.sub(r'"\s*:', '":', json_str)    # Fix spacing after quotes
        
        # Clean up whitespace
        json_str = re.sub(r'\s+', ' ', json_str)
        json_str = json_str.strip()
        
        return json_str
#####


    
    def _extract_values_manually(self, json_str):
        """Extract values manually as last resort"""
        try:
            # Extract alpha
            alpha_match = re.search(r'"alpha"\s*:\s*(\d+)', json_str)
            alpha = int(alpha_match.group(1)) if alpha_match else 40
            
            # Extract kappa
            kappa_match = re.search(r'"kappa"\s*:\s*([\d.]+)', json_str)
            kappa = float(kappa_match.group(1)) if kappa_match else 0.15
            
            # Extract L
            L_match = re.search(r'"L"\s*:\s*(\d+)', json_str)
            L = int(L_match.group(1)) if L_match else 6
            
            # Extract runtime_threshold
            runtime_match = re.search(r'"runtime_threshold"\s*:\s*([\d.]+)', json_str)
            runtime = float(runtime_match.group(1)) if runtime_match else 7.0
            
            # Extract search_intensity
            intensity_match = re.search(r'"search_intensity"\s*:\s*["\'](\w+)["\']', json_str)
            intensity = intensity_match.group(1) if intensity_match else 'medium'
            
            # Extract operator_strategy
            operators = []
            operator_match = re.search(r'"operator_strategy"\s*:\s*\[(.*?)\]', json_str, re.DOTALL)
            if operator_match:
                ops_str = operator_match.group(1)
                op_matches = re.findall(r'["\'](\w+)["\']', ops_str)
                operators = op_matches if op_matches else ['swap', 'greedy_add']
            else:
                operators = ['swap', 'greedy_add']
            
            # Extract reasoning
            reasoning_match = re.search(r'"reasoning"\s*:\s*["\']([^"\']*)["\']', json_str)
            reasoning = reasoning_match.group(1) if reasoning_match else "Extracted manually"
            
            return {
                'alpha': alpha,
                'kappa': kappa,
                'L': L,
                'operator_strategy': operators,
                'runtime_threshold': runtime,
                'search_intensity': intensity,
                'reasoning': reasoning
            }
        except Exception as e:
            print(f"⚠️ Manual extraction failed: {e}")
            return None
    
    def generate_operators_prompt(self, instance_features, target_performance, generation):
        """Generate enhanced prompt for operator selection with stricter JSON formatting"""
        
        optimal_count = self.operator_manager._get_optimal_operator_count(instance_features)
        
        prompt = f"""You are an expert in multi-objective optimization operator selection.
    
    INSTANCE FEATURES:
    - Number of items: {instance_features['n_items']}
    - Number of objectives: {instance_features['n_objectives']}
    - Instance size: {instance_features['instance_size']}
    - Capacity tightness: {instance_features.get('capacity_tightness', 'medium')}
    - Weight-profit correlation: {instance_features.get('weight_profit_correlation', 0.0):.2f}
    - Subclass: {instance_features['subclass']}
    
    OPTIMIZATION CONTEXT:
    - Generation: {generation}
    - Target hypervolume: {target_performance:,.0f}
    - Optimal operator count: {optimal_count}
    
    AVAILABLE OPERATORS:
    1. swap: Exchange items in solution (low complexity)
    2. greedy_add: Add best feasible items (medium complexity)
    3. mutation: Random modifications (low complexity)
    4. local_search: Neighborhood exploration (high complexity)
    5. repair: Fix infeasible solutions (medium complexity)
    6. recombine: Combine solution parts (medium complexity)
    7. intensify: Intensify search around good solutions (high complexity)
    8. diversify: Explore diverse regions (medium complexity)
    
    SELECTION GUIDELINES:
    - Early generations (0-2): Prioritize exploration operators (mutation, diversify, recombine)
    - Middle generations (3-5): Use balanced approach
    - Late generations (6+): Prioritize exploitation operators (local_search, intensify, greedy_add)
    - Tight capacity: Include repair operator
    - Many objectives: Include diversify operator
    - Large instances: Prefer efficient operators (greedy_add, local_search)
    
    TASK:
    Select exactly {optimal_count} operators optimized for this instance and generation.
    Consider the instance characteristics and current optimization phase.
    
    CRITICAL: Return ONLY valid JSON without any explanations outside the JSON structure.
    Do not use escape characters in your reasoning text.
    Keep the reasoning brief and simple.
    
    RESPONSE FORMAT (copy this exactly):
    {{
        "alpha": 40,
        "kappa": 0.150,
        "L": 6,
        "operator_strategy": ["swap", "greedy_add", "local_search"],
        "runtime_threshold": 7.0,
        "search_intensity": "medium",
        "reasoning": "Brief explanation without special characters"
    }}"""
        return prompt
    
    def initialize_evolutionary_population(self, instance_features, target_performance, generation=0):
        """Enhanced population initialization with adaptive operators"""
        
        if self.connection_status != "ollama_connected":
            return self._initialize_rule_based_population(instance_features, target_performance, generation)
        
        population = []
        
        # Check if we have knowledge for this subclass
        subclass = instance_features['subclass']
        if subclass in self.knowledge_base:
            # Use knowledge from previous runs
            base_configs = self.knowledge_base[subclass]
            for config in base_configs[:3]:  # Use top 3 configs
                # Update operators for current generation
                optimal_operators = self.operator_manager.get_optimal_operators(instance_features, generation)
                config['operator_strategy'] = optimal_operators
                population.append(config.copy())
                print(f"✅ Reused knowledge from previous run for {subclass}")
        
        # Generate configurations with different focuses
        focus_areas = [
            "exploration-focused",
            "exploitation-focused",
            "balanced",
            "runtime-efficient",
            "quality-focused"
        ]
        
        # Only generate new configs if we don't have enough from knowledge
        configs_needed = max(0, 5 - len(population))
        
        for i in range(configs_needed):
            focus = focus_areas[i % len(focus_areas)]
            
            # Use adaptive operator prompt
            prompt = self.generate_operators_prompt(instance_features, target_performance, generation)
            
            response = self._call_ollama(prompt)
            
            if response:
                config = self._extract_json_from_response(response)
                
                if config:
                    # Validate and adjust parameters
                    config['alpha'] = min(60, max(20, int(config.get('alpha', 40))))
                    config['kappa'] = min(0.25, max(0.03, float(config.get('kappa', 0.15))))
                    config['L'] = min(12, max(2, int(config.get('L', 6))))
                    config['runtime_threshold'] = min(15.0, max(5.0, float(config.get('runtime_threshold', 7.0))))
                    
                    # Validate operators
                    valid_operators = list(self.operator_manager.available_operators.keys())
                    strategy = config.get('operator_strategy', ['swap', 'greedy_add'])
                    if isinstance(strategy, str):
                        strategy = [strategy]
                    config['operator_strategy'] = [op for op in strategy if op in valid_operators]
                    
                    if not config['operator_strategy']:
                        config['operator_strategy'] = ['swap', 'greedy_add', 'mutation']
                    
                    # Validate search intensity
                    if config.get('search_intensity') not in ['low', 'medium', 'high']:
                        config['search_intensity'] = 'medium'
                    
                    config['focus'] = focus
                    population.append(config)
                    print(f"✅ Successfully generated {focus} config from LLM")
                else:
                    print(f"⚠️ Could not parse JSON from LLM response for {focus} config")
                    # Add fallback config
                    population.append(self._generate_single_rule_based_config(instance_features, target_performance, focus, generation))
            else:
                print(f"⚠️ No response from LLM for {focus} config")
                # Add fallback config
                population.append(self._generate_single_rule_based_config(instance_features, target_performance, focus, generation))
        
        # Ensure we have at least 5 configs
        while len(population) < 5:
            population.append(self._generate_single_rule_based_config(instance_features, target_performance, 'balanced', generation))
        
        self.population_configs = population
        return population
    
    def _generate_single_rule_based_config(self, instance_features, target_performance, focus='balanced', generation=0):
        """Generate a single rule-based configuration with adaptive operators"""
        
        # Get optimal operators for this instance and generation
        optimal_operators = self.operator_manager.get_optimal_operators(instance_features, generation)
        
        # Base parameters on instance size and focus
        if instance_features['instance_size'] == 'small':
            if focus == 'exploration-focused':
                alpha = np.random.choice([35, 40, 45])
                kappa = np.random.choice([0.08, 0.10, 0.12])
                L = np.random.choice([5, 6, 7])
                runtime = 8.0
            elif focus == 'exploitation-focused':
                alpha = np.random.choice([25, 30, 35])
                kappa = np.random.choice([0.12, 0.15, 0.18])
                L = np.random.choice([4, 5, 6])
                runtime = 6.0
            elif focus == 'runtime-efficient':
                alpha = np.random.choice([20, 25, 30])
                kappa = np.random.choice([0.15, 0.18, 0.20])
                L = np.random.choice([3, 4, 5])
                runtime = 5.0
            elif focus == 'quality-focused':
                alpha = np.random.choice([45, 50, 55])
                kappa = np.random.choice([0.05, 0.08, 0.10])
                L = np.random.choice([7, 8, 9])
                runtime = 10.0
            else:  # balanced
                alpha = np.random.choice([30, 35, 40])
                kappa = np.random.choice([0.10, 0.12, 0.15])
                L = np.random.choice([4, 5, 6])
                runtime = 7.0
        elif instance_features['instance_size'] == 'medium':
            if focus == 'exploration-focused':
                alpha = np.random.choice([45, 50, 55])
                kappa = np.random.choice([0.10, 0.12, 0.15])
                L = np.random.choice([7, 8, 9])
                runtime = 12.0
            elif focus == 'exploitation-focused':
                alpha = np.random.choice([35, 40, 45])
                kappa = np.random.choice([0.15, 0.18, 0.20])
                L = np.random.choice([6, 7, 8])
                runtime = 10.0
            elif focus == 'runtime-efficient':
                alpha = np.random.choice([30, 35, 40])
                kappa = np.random.choice([0.18, 0.20, 0.22])
                L = np.random.choice([5, 6, 7])
                runtime = 8.0
            elif focus == 'quality-focused':
                alpha = np.random.choice([55, 60, 65])
                kappa = np.random.choice([0.08, 0.10, 0.12])
                L = np.random.choice([9, 10, 11])
                runtime = 15.0
            else:  # balanced
                alpha = np.random.choice([40, 45, 50])
                kappa = np.random.choice([0.12, 0.15, 0.18])
                L = np.random.choice([6, 7, 8])
                runtime = 10.0
        else:  # large
            if focus == 'exploration-focused':
                alpha = np.random.choice([50, 55, 60])
                kappa = np.random.choice([0.12, 0.15, 0.18])
                L = np.random.choice([8, 9, 10])
                runtime = 15.0
            elif focus == 'exploitation-focused':
                alpha = np.random.choice([40, 45, 50])
                kappa = np.random.choice([0.18, 0.20, 0.22])
                L = np.random.choice([7, 8, 9])
                runtime = 12.0
            elif focus == 'runtime-efficient':
                alpha = np.random.choice([35, 40, 45])
                kappa = np.random.choice([0.20, 0.22, 0.25])
                L = np.random.choice([6, 7, 8])
                runtime = 10.0
            elif focus == 'quality-focused':
                alpha = np.random.choice([60, 65, 70])
                kappa = np.random.choice([0.10, 0.12, 0.15])
                L = np.random.choice([10, 11, 12])
                runtime = 20.0
            else:  # balanced
                alpha = np.random.choice([45, 50, 55])
                kappa = np.random.choice([0.15, 0.18, 0.20])
                L = np.random.choice([7, 8, 9])
                runtime = 12.0
        
        # Adjust based on capacity tightness
        if instance_features.get('capacity_tightness') == 'very_tight':
            alpha = min(60, alpha + 5)
            L = min(12, L + 1)
        elif instance_features.get('capacity_tightness') == 'loose':
            alpha = max(20, alpha - 5)
            L = max(2, L - 1)
        
        return {
            'alpha': alpha,
            'kappa': kappa,
            'L': L,
            'operator_strategy': optimal_operators,
            'runtime_threshold': runtime,
            'search_intensity': 'medium',
            'reasoning': f'Rule-based {focus} config',
            'focus': focus
        }
    
    def _initialize_rule_based_population(self, instance_features, target_performance, generation=0):
        """Initialize population with rule-based configurations"""
        population = []
        
        # Generate diverse configurations with different focuses
        focus_areas = ['exploration-focused', 'exploitation-focused', 'balanced', 'runtime-efficient', 'quality-focused']
        
        for focus in focus_areas:
            config = self._generate_single_rule_based_config(instance_features, target_performance, focus, generation)
            population.append(config)
        
        self.population_configs = population
        return population
    
    def evolutionary_crossover(self, parent1, parent2, instance_features, generation=0):
        """Enhanced crossover with adaptive operator selection"""
        
        if self.connection_status != "ollama_connected":
            return self._rule_based_crossover(parent1, parent2, instance_features, generation)
        
        prompt = f"""
You are performing evolutionary crossover for multi-objective optimization.

INSTANCE FEATURES:
- Number of items: {instance_features['n_items']}
- Number of objectives: {instance_features['n_objectives']}
- Instance size: {instance_features['instance_size']}
- Capacity tightness: {instance_features.get('capacity_tightness', 'medium')}
- Weight-profit correlation: {instance_features.get('weight_profit_correlation', 0.0):.2f}
- Subclass: {instance_features['subclass']}

PARENT CONFIGURATIONS:
Parent 1: {parent1}
Parent 2: {parent2}

OPTIMIZATION CONTEXT:
- Generation: {generation}

TASK:
Create a CHILD configuration by combining the best aspects of both parents.
The child should inherit beneficial traits from both parents while maintaining diversity.
Focus on combining the strengths of both parents for this specific instance subclass.
Select operators appropriate for the current generation.

IMPORTANT: Return ONLY valid JSON.

RESPONSE FORMAT:
{{
    "alpha": 40,
    "kappa": 0.150,
    "L": 6,
    "operator_strategy": ["swap", "greedy_add", "local_search"],
    "runtime_threshold": 7.0,
    "search_intensity": "medium",
    "reasoning": "Brief explanation of crossover strategy"
}}
"""
        
        response = self._call_ollama(prompt)
        
        if response:
            child = self._extract_json_from_response(response)
            
            if child:
                # Validate and adjust parameters
                child['alpha'] = min(60, max(20, int(child.get('alpha', 40))))
                child['kappa'] = min(0.25, max(0.03, float(child.get('kappa', 0.15))))
                child['L'] = min(12, max(2, int(child.get('L', 6))))
                child['runtime_threshold'] = min(15.0, max(5.0, float(child.get('runtime_threshold', 7.0))))
                
                # Validate operators
                valid_operators = list(self.operator_manager.available_operators.keys())
                strategy = child.get('operator_strategy', ['swap', 'greedy_add'])
                if isinstance(strategy, str):
                    strategy = [strategy]
                child['operator_strategy'] = [op for op in strategy if op in valid_operators]
                
                if not child['operator_strategy']:
                    child['operator_strategy'] = ['swap', 'greedy_add', 'mutation']
                
                # Validate search intensity
                if child.get('search_intensity') not in ['low', 'medium', 'high']:
                    child['search_intensity'] = 'medium'
                
                return child
        
        # Fallback to rule-based crossover
        return self._rule_based_crossover(parent1, parent2, instance_features, generation)
    
    def _rule_based_crossover(self, parent1, parent2, instance_features, generation=0):
        """Enhanced rule-based crossover with adaptive operator selection"""
        
        child = {}
        
        # Crossover alpha (weighted average based on parent performance if available)
        if 'performance' in parent1 and 'performance' in parent2:
            weight1 = parent1['performance'] / (parent1['performance'] + parent2['performance'])
            weight2 = 1 - weight1
            child['alpha'] = int(parent1['alpha'] * weight1 + parent2['alpha'] * weight2)
        else:
            child['alpha'] = int((parent1['alpha'] + parent2['alpha']) / 2)
        
        # Crossover kappa (weighted average)
        if 'performance' in parent1 and 'performance' in parent2:
            child['kappa'] = parent1['kappa'] * weight1 + parent2['kappa'] * weight2
        else:
            child['kappa'] = (parent1['kappa'] + parent2['kappa']) / 2
        
        # Crossover L (weighted average)
        if 'performance' in parent1 and 'performance' in parent2:
            child['L'] = int(parent1['L'] * weight1 + parent2['L'] * weight2)
        else:
            child['L'] = int((parent1['L'] + parent2['L']) / 2)
        
        # Crossover operators (union of parent operators, limited to optimal count)
        operators1 = set(parent1['operator_strategy'])
        operators2 = set(parent2['operator_strategy'])
        child_operators = list(operators1.union(operators2))
        
        # Get optimal operators for current generation
        optimal_operators = self.operator_manager.get_optimal_operators(instance_features, generation)
        
        # Combine parent operators with optimal operators
        all_operators = list(set(child_operators + optimal_operators))
        
        # Select the best operators based on performance
        operator_scores = []
        for op in all_operators:
            score = self.operator_manager.operator_performance.get(op, {}).get('success_rate', 0.5)
            if op in optimal_operators:
                score *= 1.2  # Boost optimal operators
            operator_scores.append((op, score))
        
        operator_scores.sort(key=lambda x: x[1], reverse=True)
        optimal_count = self.operator_manager._get_optimal_operator_count(instance_features)
        child['operator_strategy'] = [op for op, _ in operator_scores[:optimal_count]]
        
        if len(child['operator_strategy']) < 2:
            child['operator_strategy'] = ['swap', 'greedy_add', 'mutation']
        
        # Crossover runtime threshold (weighted average)
        if 'performance' in parent1 and 'performance' in parent2:
            child['runtime_threshold'] = parent1['runtime_threshold'] * weight1 + parent2['runtime_threshold'] * weight2
        else:
            child['runtime_threshold'] = (parent1['runtime_threshold'] + parent2['runtime_threshold']) / 2
        
        # Crossover search intensity (choose from parents)
        child['search_intensity'] = parent1['search_intensity'] if np.random.random() > 0.5 else parent2['search_intensity']
        
        # Adjust parameters based on instance features
        if instance_features.get('capacity_tightness') == 'very_tight':
            child['alpha'] = min(60, child['alpha'] + 2)
            child['L'] = min(12, child['L'] + 1)
        elif instance_features.get('capacity_tightness') == 'loose':
            child['alpha'] = max(20, child['alpha'] - 2)
            child['L'] = max(2, child['L'] - 1)
        
        child['reasoning'] = f"Rule-based crossover of parents"
        
        return child
    
    def evolutionary_mutation(self, config, instance_features, mutation_strength='medium', generation=0):
        """Enhanced mutation with adaptive operator selection"""
        
        if self.connection_status != "ollama_connected":
            return self._rule_based_mutation(config, instance_features, mutation_strength, generation)
        
        prompt = f"""
You are performing evolutionary mutation for multi-objective optimization.

INSTANCE FEATURES:
- Number of items: {instance_features['n_items']}
- Number of objectives: {instance_features['n_objectives']}
- Instance size: {instance_features['instance_size']}
- Capacity tightness: {instance_features.get('capacity_tightness', 'medium')}
- Weight-profit correlation: {instance_features.get('weight_profit_correlation', 0.0):.2f}
- Subclass: {instance_features['subclass']}

CURRENT CONFIGURATION:
{config}

OPTIMIZATION CONTEXT:
- Generation: {generation}
- Mutation strength: {mutation_strength}

TASK:
Create a MUTATED configuration by introducing controlled random changes.
The mutation should explore new regions while maintaining the core strengths.
Focus on adapting the configuration to better suit this specific instance subclass.
Select operators appropriate for the current generation and mutation strength.

IMPORTANT: Return ONLY valid JSON.

RESPONSE FORMAT:
{{
    "alpha": 40,
    "kappa": 0.150,
    "L": 6,
    "operator_strategy": ["swap", "greedy_add", "local_search"],
    "runtime_threshold": 7.0,
    "search_intensity": "medium",
    "reasoning": "Brief explanation of mutation strategy"
}}
"""
        
        response = self._call_ollama(prompt)
        
        if response:
            mutated = self._extract_json_from_response(response)
            
            if mutated:
                # Validate and adjust parameters
                mutated['alpha'] = min(60, max(20, int(mutated.get('alpha', 40))))
                mutated['kappa'] = min(0.25, max(0.03, float(mutated.get('kappa', 0.15))))
                mutated['L'] = min(12, max(2, int(mutated.get('L', 6))))
                mutated['runtime_threshold'] = min(15.0, max(5.0, float(mutated.get('runtime_threshold', 7.0))))
                
                # Validate operators
                valid_operators = list(self.operator_manager.available_operators.keys())
                strategy = mutated.get('operator_strategy', ['swap', 'greedy_add'])
                if isinstance(strategy, str):
                    strategy = [strategy]
                mutated['operator_strategy'] = [op for op in strategy if op in valid_operators]
                
                if not mutated['operator_strategy']:
                    mutated['operator_strategy'] = ['swap', 'greedy_add', 'mutation']
                
                # Validate search intensity
                if mutated.get('search_intensity') not in ['low', 'medium', 'high']:
                    mutated['search_intensity'] = 'medium'
                
                return mutated
        
        # Fallback to rule-based mutation
        return self._rule_based_mutation(config, instance_features, mutation_strength, generation)
    
    def _rule_based_mutation(self, config, instance_features, mutation_strength='medium', generation=0):
        """Enhanced rule-based mutation with adaptive operator selection"""
        
        mutated = config.copy()
        
        # Determine mutation magnitude based on strength
        if mutation_strength == 'low':
            alpha_mutation = np.random.choice([-3, 0, 3])
            kappa_mutation = np.random.choice([-0.02, 0, 0.02])
            L_mutation = np.random.choice([-1, 0, 1])
        elif mutation_strength == 'high':
            alpha_mutation = np.random.choice([-8, -5, 0, 5, 8])
            kappa_mutation = np.random.choice([-0.05, -0.03, 0, 0.03, 0.05])
            L_mutation = np.random.choice([-2, -1, 0, 1, 2])
        else:  # medium
            alpha_mutation = np.random.choice([-5, -3, 0, 3, 5])
            kappa_mutation = np.random.choice([-0.03, -0.02, 0, 0.02, 0.03])
            L_mutation = np.random.choice([-1, 0, 1])
        
        # Apply mutations
        mutated['alpha'] = min(60, max(20, config['alpha'] + alpha_mutation))
        mutated['kappa'] = min(0.25, max(0.03, config['kappa'] + kappa_mutation))
        mutated['L'] = min(12, max(2, config['L'] + L_mutation))
        
        # Mutate operators with some probability
        if np.random.random() < 0.3:
            # Get optimal operators for current generation
            optimal_operators = self.operator_manager.get_optimal_operators(instance_features, generation)
            
            # Replace one operator with a random optimal operator
            current_operators = config['operator_strategy'].copy()
            if len(current_operators) > 0:
                removed = np.random.choice(current_operators)
                current_operators.remove(removed)
            
            available_optimal = [op for op in optimal_operators if op not in current_operators]
            if available_optimal:
                added = np.random.choice(available_optimal)
                current_operators.append(added)
            
            mutated['operator_strategy'] = current_operators
        
        # Mutate search intensity with some probability
        if np.random.random() < 0.2:
            intensities = ['low', 'medium', 'high']
            current_idx = intensities.index(config['search_intensity'])
            new_idx = (current_idx + np.random.choice([-1, 1])) % 3
            mutated['search_intensity'] = intensities[new_idx]
        
        # Adjust parameters based on instance features
        if instance_features.get('capacity_tightness') == 'very_tight':
            mutated['alpha'] = min(60, mutated['alpha'] + 2)
            mutated['L'] = min(12, mutated['L'] + 1)
        elif instance_features.get('capacity_tightness') == 'loose':
            mutated['alpha'] = max(20, mutated['alpha'] - 2)
            mutated['L'] = max(2, mutated['L'] - 1)
        
        mutated['reasoning'] = f"Rule-based {mutation_strength} mutation"
        
        return mutated
    
    def update_knowledge_base(self, subclass, configs, performances):
        """Update knowledge base with successful configurations"""
        if subclass not in self.knowledge_base:
            self.knowledge_base[subclass] = []
        
        # Sort configs by performance
        sorted_configs = sorted(zip(configs, performances), key=lambda x: x[1], reverse=True)
        
        # Keep only top 5 configs
        self.knowledge_base[subclass] = [config for config, _ in sorted_configs[:5]]
        
        print(f"✅ Updated knowledge base for {subclass} with {len(self.knowledge_base[subclass])} configs")

In [86]:
def test_json_parsing():
    """Test the improved JSON parsing"""
    test_response = '''Here's my response:
{
    "alpha": 40,
    "kappa": 0.150,
    "L": 6,
    "operator_strategy": ["swap", "greedy_add", "local_search"],
    "runtime_threshold": 7.0,
    "search_intensity": "medium",
    "reasoning": "This is a test configuration"
}'''

    # Create a minimal test class with just the parsing method
    class TestJSONParser:
        def _extract_json_from_response(self, response):
            """Ultra-robust JSON extraction from LLM response"""
            if not response:
                return None
                
            # More aggressive preprocessing to fix all escape issues
            response = response.replace('\\', '\\\\')  # Fix backslashes
            response = response.replace('\n', ' ')     # Remove newlines
            response = response.replace('\r', '')     # Remove carriage returns
            response = response.replace('\t', ' ')     # Remove tabs
            
            # Fix all types of escape sequences using string replacement instead of regex
            response = response.replace('\\n', ' ')  # Fix escaped newlines
            response = response.replace('\\r', ' ')  # Fix escaped carriage returns
            response = response.replace('\\t', ' ')  # Fix escaped tabs
            response = response.replace('\\"', '"')  # Fix escaped quotes
            
            # Try multiple extraction methods in order of reliability
            json_str = None
            
            # Method 1: Look for JSON code blocks first (most reliable)
            pattern = r'```(?:json)?\s*(\{.*?\})\s*```'
            match = re.search(pattern, response, re.DOTALL)
            if match:
                json_str = match.group(1)
            
            # Method 2: Look for JSON between first { and last }
            if json_str is None:
                start_idx = response.find('{')
                end_idx = response.rfind('}') + 1
                if start_idx != -1 and end_idx != -1:
                    json_str = response[start_idx:end_idx]
            
            # Method 3: Try to find any JSON-like structure
            if json_str is None:
                pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
                matches = re.findall(pattern, response, re.DOTALL)
                if matches:
                    json_str = matches[0]  # Take the first match
            
            if json_str:
                # Apply comprehensive JSON fixing
                json_str = self._fix_json_issues_comprehensive(json_str)
                
                try:
                    parsed_json = json.loads(json_str)
                    return parsed_json
                except json.JSONDecodeError as e:
                    print(f"⚠️ JSON parsing error: {e}")
                    print(f"Problematic JSON: {json_str[:200]}...")
                    return None
            
            return None
        
        def _fix_json_issues_comprehensive(self, json_str):
            """Comprehensive JSON fixing for all types of formatting issues"""
            # Fix broken string concatenations (like "medium""reasoning")
            json_str = re.sub(r'"\s*"\s*', '', json_str)
            
            # Fix missing commas between objects
            json_str = re.sub(r'}\s*{', '},{', json_str)
            json_str = re.sub(r'"\s*}\s*"', '","', json_str)
            json_str = re.sub(r'"\s*\]\s*"', '","', json_str)
            
            # Fix missing commas in arrays
            json_str = re.sub(r'"\s*"\s*', '","', json_str)
            
            # Fix trailing commas
            json_str = re.sub(r',\s*}', '}', json_str)
            json_str = re.sub(r',\s*]', ']', json_str)
            
            # Fix unescaped quotes in values
            json_str = re.sub(r':\s*"([^"]*)"([^"]*?)"', r': "\1\\"\\2\\""', json_str)
            
            # Fix broken key-value pairs
            json_str = re.sub(r'(\w+)\s*:\s*"', r'"\1": "', json_str)
            
            # Fix common LLM output issues
            json_str = re.sub(r':\s*"', ':"', json_str)  # Fix spacing before quotes
            json_str = re.sub(r'"\s*:', '":', json_str)    # Fix spacing after quotes
            
            # Clean up whitespace
            json_str = re.sub(r'\s+', ' ', json_str)
            json_str = json_str.strip()
            
            return json_str
    
    # Test the parsing
    parser = TestJSONParser()
    result = parser._extract_json_from_response(test_response)
    print("Test result:", result)
    
    # Test with problematic JSON
    problematic_response = '''Here's my response:
{
    "alpha": 40,
    "kappa": 0.150,
    "L": 6,
    "operator_strategy": ["swap", "greedy_add", "local_search"],
    "runtime_threshold": 7.0,
    "search_intensity": "medium",
    "reasoning": "This is a test with \"quotes\" and \\backslashes\\"
}'''
    
    result2 = parser._extract_json_from_response(problematic_response)
    print("Test result with problematic JSON:", result2)

# Run the test
test_json_parsing()

⚠️ JSON parsing error: Invalid \escape: line 1 column 155 (char 154)
Problematic JSON: { "alpha": 40, "kappa": 0.150, "L": 6, "operator_strategy": ["swap", "greedy_add", "local_search"], "runtime_threshold": 7.0, "search_intensity":"medium\"\2\""reasoning":"This is a test configuration"...
Test result: None
⚠️ JSON parsing error: Invalid \escape: line 1 column 155 (char 154)
Problematic JSON: { "alpha": 40, "kappa": 0.150, "L": 6, "operator_strategy": ["swap", "greedy_add", "local_search"], "runtime_threshold": 7.0, "search_intensity":"medium\"\2\""reasoning":"This is a test with \"\2\"" a...
Test result with problematic JSON: None


In [87]:
# Test the improved JSON parsing
test_response = '''Here's my response:
{
    "alpha": 40,
    "kappa": 0.150,
    "L": 6,
    "operator_strategy": ["swap", "greedy_add", "local_search"],
    "runtime_threshold": 7.0,
    "search_intensity": "medium",
    "reasoning": "This is a test configuration"
}'''

# Create a test instance
test_llm = MindEvolutionLLMInterface()
result = test_llm._extract_json_from_response(test_response)
print("Test result:", result)

✅ Ollama connected with llama3:latest available
Test result: {'alpha': 40, 'kappa': 0.15, 'L': 6, 'operator_strategy': ['swap', 'greedy_add', 'local_search'], 'runtime_threshold': 7.0, 'search_intensity': 'medium', 'reasoning': 'This is a test configuration'}


In [88]:
%%cython
"""
Enhanced MOACP Implementation with All 8 Operators and Adaptive Selection
"""

from libc.stdlib cimport malloc, free, srand, rand
from libc.string cimport memset
from libc.math cimport exp
import numpy as np
import time

# Structs
cdef struct ind:
    int nombr_nonpris
    int nombr
    int rank
    float fitnessbest
    float fitness
    int explored
    double *f
    double *capa
    double *v
    int *d
    int *Items

cdef struct pop:
    int size
    int maxsize
    ind **ind_array

# Enhanced agent struct for Mind Evolution
cdef struct agent:
    int agent_id
    double performance_score
    int generation
    int parent1_id
    int parent2_id
    bint is_mutant
    double *config_params

# Globals
cdef int NBITEMS = 250
cdef int ni = 250
cdef int L = 5
cdef double LARGE = 10e50
cdef float smallValue = 0.0000001
cdef double kappa = 0.05
cdef int alpha = 10
cdef int paretoIni = 28000

cdef int nf = 2
cdef double *capacities = NULL
cdef int **weights = NULL
cdef int **profits = NULL
cdef double *vector_weight = NULL
cdef double max_bound = 0.0
cdef double **OBJ_Weights = NULL
cdef int nombreLIGNE = 0
cdef int nextLn = 0
cdef int inv = 0
cdef int OBJ_Weights_lines = 0

# Agent population for Mind Evolution
cdef agent *agent_population = NULL
cdef int num_agents = 5

def seed(int x):
    srand(x)

cdef int irand(int range_val):
    return rand() % range_val

cdef void *chk_malloc(size_t size):
    cdef void *return_value = malloc(size)
    if return_value == NULL:
        raise MemoryError("Out of memory.")
    memset(return_value, 0, size)
    return return_value

cdef pop *create_pop(int maxsize, int nf):
    cdef int i
    cdef pop *pp = <pop *>chk_malloc(sizeof(pop))
    pp.size = 0
    pp.maxsize = maxsize
    pp.ind_array = <ind **>chk_malloc(maxsize * sizeof(void*))
    for i in range(maxsize):
        pp.ind_array[i] = NULL
    return pp

cdef ind *create_ind(int nf):
    cdef int i
    cdef ind *p_ind = <ind *>chk_malloc(sizeof(ind))
    p_ind.nombr_nonpris = 0
    p_ind.nombr = 0
    p_ind.rank = 0
    p_ind.fitnessbest = -1.0
    p_ind.fitness = -1.0
    p_ind.explored = 0
    p_ind.f = <double *>chk_malloc(nf * sizeof(double))
    p_ind.capa = <double *>chk_malloc(nf * sizeof(double))
    p_ind.v = <double *>chk_malloc(nf * sizeof(double))
    p_ind.d = <int *>chk_malloc(ni * sizeof(int))
    p_ind.Items = <int *>chk_malloc(ni * sizeof(int))
    for i in range(ni):
        p_ind.Items[i] = 0
        p_ind.d[i] = 0
    for i in range(nf):
        p_ind.f[i] = 0.0
        p_ind.capa[i] = 0.0
        p_ind.v[i] = 0.0
    return p_ind

cdef ind *ind_copy(ind *i):
    cdef ind *p_ind = create_ind(nf)
    cdef int k
    p_ind.nombr_nonpris = i.nombr_nonpris
    p_ind.nombr = i.nombr
    p_ind.rank = i.rank
    p_ind.fitnessbest = i.fitnessbest
    p_ind.fitness = i.fitness
    p_ind.explored = i.explored
    for k in range(nf):
        p_ind.f[k] = i.f[k]
        p_ind.v[k] = i.v[k]
        p_ind.capa[k] = i.capa[k]
    for k in range(ni):
        p_ind.d[k] = i.d[k]
        p_ind.Items[k] = i.Items[k]
    return p_ind

cdef void free_ind(ind *p_ind):
    if p_ind != NULL:
        free(p_ind.d)
        free(p_ind.f)
        free(p_ind.capa)
        free(p_ind.v)
        free(p_ind.Items)
        free(p_ind)

cdef void complete_free_pop(pop *pp):
    cdef int i
    if pp != NULL:
        if pp.ind_array != NULL:
            for i in range(pp.size):
                if pp.ind_array[i] != NULL:
                    free_ind(pp.ind_array[i])
                    pp.ind_array[i] = NULL
            free(pp.ind_array)
        free(pp)

cdef void cleanup_globals():
    global capacities, weights, profits, vector_weight, OBJ_Weights, OBJ_Weights_lines, nf, ni
    if capacities != NULL:
        free(capacities)
        capacities = NULL
    if weights != NULL:
        for i in range(nf):
            if weights[i] != NULL:
                free(weights[i])
        free(weights)
        weights = NULL
    if profits != NULL:
        for i in range(nf):
            if profits[i] != NULL:
                free(profits[i])
        free(profits)
        profits = NULL
    if vector_weight != NULL:
        free(vector_weight)
        vector_weight = NULL
    if OBJ_Weights != NULL:
        for i in range(nf):
            if OBJ_Weights[i] != NULL:
                free(OBJ_Weights[i])
        free(OBJ_Weights)
        OBJ_Weights = NULL
    OBJ_Weights_lines = 0
    nf = 0
    ni = 0

cdef int non_dominated(ind *p_ind_a, ind *p_ind_b):
    cdef int i
    cdef int a_is_good = -1
    cdef int equal = 1
    for i in range(nf):
        if p_ind_a.f[i] > p_ind_b.f[i]:
            a_is_good = 1
        if p_ind_a.f[i] != p_ind_b.f[i]:
            equal = 0
    if equal:
        return 0
    return a_is_good

cdef double calcAddEpsIndicator(ind *p_ind_a, ind *p_ind_b):
    global max_bound
    cdef int i
    cdef double eps
    cdef double temp_eps
    if max_bound == 0.0:
        max_bound = 1e-8
    eps = (p_ind_a.v[0]/max_bound)-(p_ind_b.v[0]/max_bound)
    for i in range(1, nf):
        temp_eps = (p_ind_a.v[i]/max_bound)-(p_ind_b.v[i]/max_bound)
        if temp_eps > eps:
            eps = temp_eps
    return eps

cdef void init_fitness(ind *x):
    x.fitness = 0.0

cdef void update_fitness(ind *x, double I):
    x.fitness -= exp(-I / kappa)

cdef double update_fitness_return(double f, double I):
    return f - exp(-I / kappa)

cdef int delete_fitness(ind *x, double I):
    x.fitness += exp(-I / kappa)
    return 0

cdef void compute_ind_fitness(ind *x, pop *SP):
    cdef int j
    init_fitness(x)
    for j in range(SP.size):
        if SP.ind_array[j] != x:
            update_fitness(x, calcAddEpsIndicator(SP.ind_array[j], x))

cdef void compute_all_fitness(pop *SP):
    cdef int i
    for i in range(SP.size):
        compute_ind_fitness(SP.ind_array[i], SP)

cdef void loadMOKP(char *filename):
    global nf, ni, capacities, weights, profits
    cdef int i, f
    with open(filename.decode(), "r") as source:
        _nf, _ni = [int(x) for x in source.readline().split()]
        nf = _nf
        ni = _ni
        capacities = <double *>chk_malloc(nf * sizeof(double))
        weights = <int **>chk_malloc(nf * sizeof(void*))
        profits = <int **>chk_malloc(nf * sizeof(void*))
        for f in range(nf):
            capacities[f] = float(source.readline().strip())
            weights[f] = <int *>chk_malloc(ni * sizeof(int))
            profits[f] = <int *>chk_malloc(ni * sizeof(int))
            for i in range(ni):
                source.readline()  # item index (ignore)
                weights[f][i] = int(source.readline().strip())
                profits[f][i] = int(source.readline().strip())

cdef void read_weights_file(char *filename):
    global OBJ_Weights, nombreLIGNE, nf, OBJ_Weights_lines
    cdef int i, j, nlines
    with open(filename.decode(), "r") as f:
        lines = [line for line in f if line.strip()]
    nlines = len(lines)
    OBJ_Weights = <double **>chk_malloc(nf * sizeof(void*))
    for i in range(nf):
        OBJ_Weights[i] = <double *>chk_malloc(nlines * sizeof(double))
    for i, line in enumerate(lines):
        vals = line.strip().split()
        for j in range(nf):
            OBJ_Weights[j][i] = float(vals[j])
    nombreLIGNE = nlines - 1
    OBJ_Weights_lines = nlines

cdef void dynamic_weight_allpop():
    global vector_weight, OBJ_Weights, nombreLIGNE, nf, nextLn
    cdef int i
    if vector_weight == NULL:
        vector_weight = <double *>chk_malloc(nf * sizeof(double))
    for i in range(nf):
        vector_weight[i] = OBJ_Weights[i][nextLn]
    if nextLn == nombreLIGNE:
        nextLn = 0
    else:
        nextLn += 1

cdef void choose_weight():
    dynamic_weight_allpop()

cdef void random_init_ind(ind *x):
    cdef int j, r, tmp
    for j in range(ni):
        x.d[j] = j
    for j in range(ni):
        r = irand(ni)
        tmp = x.d[r]
        x.d[r] = x.d[j]
        x.d[j] = tmp

cdef void evaluate(ind *x):
    cdef int j, l, k, faisable
    x.nombr = 0
    x.nombr_nonpris = 0
    for j in range(nf):
        x.capa[j] = 0.0
        x.f[j] = 0.0
    for j in range(ni):
        l = 0
        faisable = 1
        while l < nf and faisable == 1:
            if x.capa[l] + weights[l][x.d[j]] > capacities[l]:
                faisable = 0
            l += 1
        if faisable == 1:
            for k in range(nf):
                x.capa[k] += weights[k][x.d[j]]
                x.f[k] += profits[k][x.d[j]]
            x.Items[x.d[j]] = 1
            x.nombr += 1
        else:
            x.Items[x.d[j]] = 0
            x.nombr_nonpris += 1

cdef void P_init_pop(pop *SP, pop *Sarchive, int alpha):
    cdef int i, x, tmp, t
    t = max(alpha, Sarchive.size)
    cdef int* shuffle = <int *>chk_malloc(t * sizeof(int))
    for i in range(t):
        shuffle[i] = i
    for i in range(t):
        x = irand(alpha)
        tmp = shuffle[i]
        shuffle[i] = shuffle[x]
        shuffle[x] = tmp
    SP.size = alpha
    if Sarchive.size > alpha:
        for i in range(alpha):
            SP.ind_array[i] = ind_copy(Sarchive.ind_array[shuffle[i]])
    else:
        for i in range(alpha):
            if shuffle[i] < Sarchive.size:
                SP.ind_array[i] = ind_copy(Sarchive.ind_array[shuffle[i]])
            else:
                SP.ind_array[i] = create_ind(nf)
                random_init_ind(SP.ind_array[i])
                evaluate(SP.ind_array[i])
    free(shuffle)

cdef int extractPtoArchive(pop *P, pop *archive):
    cdef int i, j, dom, t, convergence_rate
    t = archive.size + P.size
    archiveAndP = create_pop(t, nf)
    convergence_rate = 0
    for i in range(archive.size):
        archiveAndP.ind_array[i] = archive.ind_array[i]
    for i in range(P.size):
        archiveAndP.ind_array[i + archive.size] = ind_copy(P.ind_array[i])
    archiveAndP.size = t
    archive.size = 0
    for i in range(t):
        for j in range(t):
            if i != j:
                dom = non_dominated(archiveAndP.ind_array[i], archiveAndP.ind_array[j])
                if dom == -1 or (dom == 0 and i > j):
                    break
        else:
            archive.ind_array[archive.size] = ind_copy(archiveAndP.ind_array[i])
            archive.size += 1
            if i >= t - P.size:
                convergence_rate += 1
    complete_free_pop(archiveAndP)
    return convergence_rate

cdef double calcMaxbound(pop *SP, int size):
    global max_bound
    cdef int i, j
    SP.size = size
    cdef double max_b = SP.ind_array[0].v[0]
    for i in range(SP.size):
        for j in range(nf):
            if max_b < SP.ind_array[i].v[j]:
                max_b = SP.ind_array[i].v[j]
    if max_b == 0.0:
        max_b = 1e-8
    max_bound = max_b
    return max_b

cdef void calcul_weight(pop *SP, int size):
    cdef int i, j
    for i in range(SP.size):
        for j in range(nf):
            SP.ind_array[i].v[j] = SP.ind_array[i].f[j] * vector_weight[j]

cdef int compute_fitness_and_select(pop *SP, ind *x, int size):
    cdef int i, worst
    cdef double worst_fit, fit_tmp
    SP.size = size
    x.fitness = 0
    compute_ind_fitness(x, SP)
    worst_fit = x.fitness
    worst = -1
    for i in range(SP.size):
        fit_tmp = update_fitness_return(SP.ind_array[i].fitness, calcAddEpsIndicator(x, SP.ind_array[i]))
        if fit_tmp > worst_fit:
            worst = i
            worst_fit = fit_tmp
    fit_tmp = x.fitness
    if worst == -1:
        return -1
    else:
        for i in range(SP.size):
            delete_fitness(SP.ind_array[i], calcAddEpsIndicator(SP.ind_array[worst], SP.ind_array[i]))
            update_fitness(SP.ind_array[i], calcAddEpsIndicator(x, SP.ind_array[i]))
        delete_fitness(x, calcAddEpsIndicator(SP.ind_array[worst], x))
        free_ind(SP.ind_array[worst])
        SP.ind_array[worst] = ind_copy(x)
        if fit_tmp - worst_fit > smallValue:
            return worst
        else:
            return -1

# Helper functions for different operators
cdef ind *apply_swap(ind *x):
    """Apply swap operator: exchange two items"""
    cdef int idx1, idx2, tmp
    idx1 = irand(ni)
    idx2 = irand(ni)
    while idx2 == idx1:
        idx2 = irand(ni)
    
    # Swap in the decision vector
    tmp = x.d[idx1]
    x.d[idx1] = x.d[idx2]
    x.d[idx2] = tmp
    
    return x

cdef ind *apply_mutation(ind *x):
    """Apply mutation operator: random modifications"""
    cdef int num_mutations, i, mutation_idx, new_pos
    
    num_mutations = max(1, ni // 10)
    for i in range(num_mutations):
        mutation_idx = irand(ni)
        # Randomly change the position of one item
        new_pos = irand(ni)
        x.d[mutation_idx] = new_pos
    
    return x

cdef ind *apply_greedy_add(ind *x, pop *SP):
    """Apply greedy add operator: add best feasible items"""
    cdef int i, j, best_idx
    cdef double best_ratio, current_ratio, total_profit, total_weight
    cdef bint feasible
    
    best_idx = -1
    best_ratio = -1.0
    
    # Find best item to add
    for i in range(ni):
        if x.Items[i] == 0:
            # Check if item can be added
            feasible = 1
            for j in range(nf):
                if x.capa[j] + weights[j][i] > capacities[j]:
                    feasible = 0
                    break
            
            if feasible:
                # Calculate profit-weight ratio
                total_profit = 0.0
                total_weight = 0.0
                for j in range(nf):
                    total_profit += profits[j][i]
                    total_weight += weights[j][i]
                
                current_ratio = total_profit / total_weight if total_weight > 0 else 0.0
                if current_ratio > best_ratio:
                    best_ratio = current_ratio
                    best_idx = i
    
    # Add the best item if found
    if best_idx != -1:
        x.Items[best_idx] = 1
        x.nombr += 1
        x.nombr_nonpris -= 1
        for j in range(nf):
            x.capa[j] += weights[j][best_idx]
            x.f[j] += profits[j][best_idx]
    
    return x

cdef ind *apply_local_search(ind *x, pop *SP):
    """Apply local search operator: neighborhood exploration"""
    cdef int i, j, k, best_neighbor
    cdef double best_improvement, current_improvement
    
    best_neighbor = -1
    best_improvement = 0.0
    
    # Try removing and adding different items
    for i in range(ni):
        if x.Items[i] == 1:
            # Try removing this item
            x.Items[i] = 0
            x.nombr -= 1
            x.nombr_nonpris += 1
            for j in range(nf):
                x.capa[j] -= weights[j][i]
                x.f[j] -= profits[j][i]
            
            # Try adding a different item
            for k in range(ni):
                if k != i and x.Items[k] == 0:
                    # Check if item k can be added
                    feasible = 1
                    for j in range(nf):
                        if x.capa[j] + weights[j][k] > capacities[j]:
                            feasible = 0
                            break
                    
                    if feasible:
                        current_improvement = 0.0
                        for j in range(nf):
                            current_improvement += profits[j][k] - profits[j][i]
                        
                        if current_improvement > best_improvement:
                            best_improvement = current_improvement
                            best_neighbor = k
            
            # Restore original item if no better neighbor found
            if best_neighbor == -1:
                x.Items[i] = 1
                x.nombr += 1
                x.nombr_nonpris -= 1
                for j in range(nf):
                    x.capa[j] += weights[j][i]
                    x.f[j] += profits[j][i]
            else:
                # Apply the best neighbor
                x.Items[best_neighbor] = 1
                x.nombr += 1
                x.nombr_nonpris -= 1
                for j in range(nf):
                    x.capa[j] += weights[j][best_neighbor]
                    x.f[j] += profits[j][best_neighbor]
                break
    
    return x

cdef ind *apply_repair(ind *x):
    """Apply repair operator: fix infeasible solutions"""
    cdef int i, j, worst_idx
    cdef bint is_feasible
    cdef double worst_ratio, current_ratio
    
    # Check feasibility
    is_feasible = 1
    for j in range(nf):
        if x.capa[j] > capacities[j]:
            is_feasible = 0
            break
    
    if is_feasible:
        return x
    
    # Remove items until feasible
    while not is_feasible:
        is_feasible = 1
        # Find item with lowest profit-weight ratio to remove
        worst_idx = -1
        worst_ratio = 1e10
        
        for i in range(ni):
            if x.Items[i] == 1:
                current_ratio = 0.0
                for j in range(nf):
                    if weights[j][i] > 0:
                        current_ratio += profits[j][i] / weights[j][i]
                
                if current_ratio < worst_ratio:
                    worst_ratio = current_ratio
                    worst_idx = i
        
        if worst_idx != -1:
            x.Items[worst_idx] = 0
            x.nombr -= 1
            x.nombr_nonpris += 1
            for j in range(nf):
                x.capa[j] -= weights[j][worst_idx]
                x.f[j] -= profits[j][worst_idx]
        
        # Check feasibility again
        for j in range(nf):
            if x.capa[j] > capacities[j]:
                is_feasible = 0
                break
    
    return x

cdef ind *apply_recombine(ind *x, pop *SP):
    """Apply recombine operator: combine solution parts"""
    cdef int parent1_idx, parent2_idx, half_point, i
    cdef ind *parent1, *parent2
    
    if SP.size < 2:
        return x
    
    # Select two random parents from the population
    parent1_idx = irand(SP.size)
    parent2_idx = irand(SP.size)
    while parent2_idx == parent1_idx:
        parent2_idx = irand(SP.size)
    
    parent1 = SP.ind_array[parent1_idx]
    parent2 = SP.ind_array[parent2_idx]
    
    # Take first half from parent1, second half from parent2
    half_point = ni // 2
    
    for i in range(half_point):
        x.d[i] = parent1.d[i]
        x.Items[i] = parent1.Items[i]
    
    for i in range(half_point, ni):
        x.d[i] = parent2.d[i]
        x.Items[i] = parent2.Items[i]
    
    # Re-evaluate the solution
    x.nombr = 0
    x.nombr_nonpris = 0
    for j in range(nf):
        x.capa[j] = 0.0
        x.f[j] = 0.0
    
    for i in range(ni):
        if x.Items[i] == 1:
            x.nombr += 1
            for j in range(nf):
                x.capa[j] += weights[j][i]
                x.f[j] += profits[j][i]
        else:
            x.nombr_nonpris += 1
    
    return x

cdef ind *apply_intensify(ind *x, pop *SP):
    """Apply intensify operator: focus around good solutions"""
    cdef int best_idx, i
    cdef double best_fitness
    cdef ind *best_solution
    
    if SP.size == 0:
        return x
    
    # Find the best solution in the population
    best_idx = 0
    best_fitness = SP.ind_array[0].fitness
    
    for i in range(1, SP.size):
        if SP.ind_array[i].fitness > best_fitness:
            best_fitness = SP.ind_array[i].fitness
            best_idx = i
    
    best_solution = SP.ind_array[best_idx]
    
    # Move current solution closer to the best solution
    for i in range(ni):
        if irand(100) < 30:  # 30% chance to copy from best solution
            x.d[i] = best_solution.d[i]
            x.Items[i] = best_solution.Items[i]
    
    # Re-evaluate
    x.nombr = 0
    x.nombr_nonpris = 0
    for j in range(nf):
        x.capa[j] = 0.0
        x.f[j] = 0.0
    
    for i in range(ni):
        if x.Items[i] == 1:
            x.nombr += 1
            for j in range(nf):
                x.capa[j] += weights[j][i]
                x.f[j] += profits[j][i]
        else:
            x.nombr_nonpris += 1
    
    return x

cdef ind *apply_diversify(ind *x):
    """Apply diversify operator: explore diverse regions"""
    cdef int i, r, tmp, num_resets, reset_idx
    
    # Randomly shuffle the decision vector
    for i in range(ni):
        r = irand(ni)
        tmp = x.d[r]
        x.d[r] = x.d[i]
        x.d[i] = tmp
    
    # Randomly reset some items
    num_resets = max(1, ni // 5)
    for i in range(num_resets):
        reset_idx = irand(ni)
        x.Items[reset_idx] = 0
    
    # Re-evaluate
    x.nombr = 0
    x.nombr_nonpris = 0
    for j in range(nf):
        x.capa[j] = 0.0
        x.f[j] = 0.0
    
    for i in range(ni):
        if x.Items[i] == 1:
            x.nombr += 1
            for j in range(nf):
                x.capa[j] += weights[j][i]
                x.f[j] += profits[j][i]
        else:
            x.nombr_nonpris += 1
    
    return x

cdef bint _is_feasible_ind(ind *x):
    """Check if a solution is feasible"""
    cdef int j
    for j in range(nf):
        if x.capa[j] > capacities[j]:
            return 0
    return 1

# Helper function to calculate current hypervolume
cdef double calculate_current_hv(pop *SP):
    """Calculate current hypervolume of the population"""
    cdef int i, j
    cdef double hv = 0.0
    cdef double ref_x = 0.0
    cdef double ref_y = 0.0
    
    if SP.size == 0:
        return 0.0
    
    # Simple hypervolume calculation for 2D
    for i in range(SP.size):
        if SP.ind_array[i].f[0] > ref_x and SP.ind_array[i].f[1] > ref_y:
            hv += (SP.ind_array[i].f[0] - ref_x) * (SP.ind_array[i].f[1] - ref_y)
    
    return hv

# Enhanced operators with all 8 operators
cdef void enhanced_mind_evolution_operator(pop *SP, pop *Sarchive, int size, double time_budget, int generation, object operator_strategy):
    """Enhanced operator with all 8 operators"""
    
    # Parse operator strategy
    cdef bint use_swap = 'swap' in operator_strategy
    cdef bint use_greedy_add = 'greedy_add' in operator_strategy
    cdef bint use_mutation = 'mutation' in operator_strategy
    cdef bint use_local_search = 'local_search' in operator_strategy
    cdef bint use_repair = 'repair' in operator_strategy
    cdef int use_recombine = 'recombine' in operator_strategy
    cdef bint use_intensify = 'intensify' in operator_strategy
    cdef bint use_diversify = 'diversify' in operator_strategy
    
    cdef ind *x, *y
    cdef int i, j, r, t, k, l, v, sol, mino, mp, maxp, consistant, pos, stop, convergence, ii, tmp_pris, tmp_nonpris, taille, feasible, tv, IM
    cdef int* remplace = <int *>chk_malloc(L * sizeof(int))
    cdef double start_time = time.time()
    cdef double current_time
    
    SP.size = size
    extractPtoArchive(SP, Sarchive)
    
    # Adapt behavior based on generation (early generations explore more)
    cdef double exploration_rate = 0.8 - (generation * 0.05)
    if exploration_rate < 0.3:
        exploration_rate = 0.3
    
    cdef int no_improvement_count = 0
    cdef double best_hv = 0.0
    cdef int iteration_count = 0
    cdef int max_iterations = 100  # Limit iterations for runtime control
    
    while (time.time() - start_time) < time_budget * 0.9 and iteration_count < max_iterations and no_improvement_count < 3:
        convergence = 0
        for i in range(SP.size):
            current_time = time.time()
            
            if (current_time - start_time) > time_budget * 0.9:
                break
                
            if not SP.ind_array[i].explored:
                x = ind_copy(SP.ind_array[i])
                j = 0
                while j < x.nombr and (current_time - start_time) < time_budget * 0.9:
                    for l in range(L):
                        remplace[l] = 0
                    
                    # Apply operators based on strategy and exploration rate
                    if use_diversify and irand(100) < (exploration_rate * 100):
                        # Diversify: explore completely different regions
                        x = apply_diversify(x)
                    elif use_recombine and irand(100) < 50:
                        # Recombine: combine parts of different solutions
                        x = apply_recombine(x, SP)
                    elif use_swap and irand(100) < 70:
                        # Swap: exchange items
                        x = apply_swap(x)
                    elif use_mutation and irand(100) < 60:
                        # Mutation: random modifications
                        x = apply_mutation(x)
                    elif use_greedy_add and irand(100) < 80:
                        # Greedy add: add best items
                        x = apply_greedy_add(x, SP)
                    elif use_intensify and irand(100) < 40:
                        # Intensify: focus around good solutions
                        x = apply_intensify(x, SP)
                    else:
                        # Local search: neighborhood exploration
                        x = apply_local_search(x, SP)
                    
                    # Apply repair if needed
                    if use_repair and not _is_feasible_ind(x):
                        x = apply_repair(x)
                    
                    # Continue with the standard replacement logic
                    mino = irand(ni)
                    while x.Items[mino] != 1 and j < x.nombr:
                        mino = irand(ni)
                    
                    if x.Items[mino] == 1:
                        x.Items[mino] = 0
                        x.nombr -= 1
                        x.nombr_nonpris += 1
                        for r in range(nf):
                            x.capa[r] -= weights[r][mino]
                            x.f[r] -= profits[r][mino]
                        
                        IM = 0
                        taille = 0
                        while IM < L and (current_time - start_time) < time_budget * 0.9:
                            while True:
                                maxp = irand(ni)
                                if x.Items[maxp] == 0:
                                    break
                            if maxp != mino:
                                consistant = 1
                                r = 0
                                while r < nf and consistant == 1:
                                    if x.capa[r] + weights[r][maxp] > capacities[r]:
                                        consistant = 0
                                    r += 1
                                if consistant == 1:
                                    feasible = 1
                                    r = 0
                                    while r < taille and feasible:
                                        if maxp == remplace[r]:
                                            feasible = 0
                                        r += 1
                                    if feasible == 1:
                                        remplace[taille] = maxp
                                        taille += 1
                                        x.Items[maxp] = 1
                                        x.nombr_nonpris -= 1
                                        x.nombr += 1
                                        for r in range(nf):
                                            x.capa[r] += weights[r][maxp]
                                            x.f[r] += profits[r][maxp]
                            IM += 1
                        
                        for tv in range(nf):
                            x.v[tv] = x.f[tv] * vector_weight[tv]
                        max_bound = calcMaxbound(SP, SP.size)
                        sol = compute_fitness_and_select(SP, x, SP.size)
                        if sol != -1:
                            j = x.nombr + 1
                            if sol > i and i + 1 < SP.size:
                                y = SP.ind_array[i + 1]
                                SP.ind_array[i + 1] = SP.ind_array[sol]
                                SP.ind_array[sol] = y
                                i += 1
                            break
                        elif sol == -1:
                            x.Items[mino] = 1
                            x.nombr_nonpris -= 1
                            x.nombr += 1
                            for r in range(nf):
                                x.capa[r] += weights[r][mino]
                                x.f[r] += profits[r][mino]
                            if taille >= 1:
                                for r in range(taille):
                                    x.Items[remplace[r]] = 0
                                    x.nombr -= 1
                                    x.nombr_nonpris += 1
                                    for t in range(nf):
                                        x.capa[t] -= weights[t][remplace[r]]
                                        x.f[t] -= profits[t][remplace[r]]
                                        x.v[t] = x.f[t] * vector_weight[t]
                    j += 1
                tmp_pris = x.nombr
                tmp_nonpris = x.nombr_nonpris
                free_ind(x)
                if j == tmp_pris:
                    SP.ind_array[i].explored = 1
        
        # Check for improvement
        current_hv = calculate_current_hv(SP)
        if current_hv <= best_hv:
            no_improvement_count += 1
        else:
            best_hv = current_hv
            no_improvement_count = 0
        
        convergence = extractPtoArchive(SP, Sarchive)
        if not convergence:
            no_improvement_count += 1
        else:
            no_improvement_count = 0
        
        iteration_count += 1
    
    free(remplace)

# Enhanced parameter configuration with agent influence
cdef void set_enhanced_parameters(
    object custom_params=None, 
    object operator_strategy=None, 
    bint print_params=True,
    int agent_id=-1
):
    global alpha, kappa, L, smallValue
    
    if custom_params is not None:
        # Apply enhanced bounds with validation
        new_alpha = int(custom_params.get('alpha', 40))
        new_kappa = float(custom_params.get('kappa', 0.15))
        new_L = int(custom_params.get('L', 6))
        new_small_value = float(custom_params.get('small_value', 1e-7))
        
        # Validate bounds
        alpha = min(60, max(20, new_alpha))
        kappa = min(0.25, max(0.03, new_kappa))
        L = min(12, max(2, new_L))
        smallValue = min(1e-6, max(1e-9, new_small_value))
        
        if print_params:
            agent_info = f" (Agent {agent_id})" if agent_id >= 0 else ""
            print(f"🚀 Enhanced Parameters{agent_info}: alpha={alpha}, kappa={kappa:.3f}, L={L}, small={smallValue:.1e}")
    else:
        alpha = 40
        kappa = 0.15
        L = 6
        smallValue = 1e-7
        if print_params:
            print(f"📊 Default Parameters: alpha={alpha}, kappa={kappa:.3f}, L={L}, small={smallValue:.1e}")
    
    if operator_strategy is not None:
        print(f"   Operator strategy: {operator_strategy}")

# Enhanced main MOACP runner with Mind Evolution
def run_moacp_mind_evolution(
    instance_file,
    weights_file,
    nbitems,
    num_objectives,
    custom_params=None,
    operator_strategy=None,
    print_params=True,
    runtime_threshold=7.0,
    num_runs=8,
    num_iterations=100,
    agent_id=-1,
    generation=0
):
    """
    Enhanced MOACP runner with Mind Evolution and adaptive operators
    """
    global nf, ni, NBITEMS, alpha, paretoIni, L, nombreLIGNE, nextLn, inv, vector_weight
    global capacities, weights, profits, OBJ_Weights

    set_enhanced_parameters(custom_params, operator_strategy, print_params, agent_id)
    NBITEMS = nbitems
    ni = nbitems
    nf = num_objectives
    paretoIni = 28000

    all_pareto_solutions = []
    run_times = []

    if print_params:
        gen_info = f" (Gen {generation})" if generation > 0 else ""
        agent_info = f" (Agent {agent_id})" if agent_id >= 0 else ""
        print(f"\n Enhanced MOACP{agent_info}{gen_info}: {num_runs} runs × {num_iterations} iterations")
        print(f" Runtime threshold: {runtime_threshold:.2f}s per run")

    total_start_time = time.time()

    for run in range(1, num_runs + 1):
        run_start_time = time.time()
        if print_params:
            print(f"   Run {run}/{num_runs}...", end=" ")

        nombreLIGNE = 0
        nextLn = 0
        inv = 0

        seed(run + agent_id * 1000)  # Unique seed per agent
        loadMOKP(instance_file.encode())
        read_weights_file(weights_file.encode())

        vector_weight = <double *>chk_malloc(nf * sizeof(double))
        P = create_pop(paretoIni, nf)

        it = 0
        while it < num_iterations:
            iteration_start = time.time()
            
            solutions = create_pop(alpha, nf)
            archive = create_pop(paretoIni, nf)
            choose_weight()
            P_init_pop(solutions, P, alpha)
            extractPtoArchive(solutions, P)
            calcul_weight(solutions, alpha)
            calcMaxbound(solutions, alpha)
            compute_all_fitness(solutions)

            # Use Enhanced Mind Evolution operator with all operators
            enhanced_mind_evolution_operator(solutions, archive, alpha, runtime_threshold * 0.6, generation, operator_strategy)

            extractPtoArchive(archive, P)
            it += 1
            complete_free_pop(solutions)
            complete_free_pop(archive)

        # Extract Pareto front for this run
        run_pareto = []
        for i in range(P.size):
            if P.ind_array[i] != NULL:
                obj1 = P.ind_array[i].f[0]
                obj2 = P.ind_array[i].f[1] if nf > 1 else 0
                run_pareto.append([obj1, obj2])

        all_pareto_solutions.extend(run_pareto)
        pareto_np = np.array(run_pareto)
        if pareto_np.shape[0] > 0:
            max_obj1 = np.max(pareto_np[:, 0])
            min_obj1 = np.min(pareto_np[:, 0])
            spread_obj1 = max_obj1 - min_obj1
            max_obj2 = np.max(pareto_np[:, 1])
            min_obj2 = np.min(pareto_np[:, 1])
            spread_obj2 = max_obj2 - min_obj2
        else:
            max_obj1 = min_obj1 = spread_obj1 = 0
            max_obj2 = min_obj2 = spread_obj2 = 0

        run_time = time.time() - run_start_time
        run_times.append(run_time)

        if print_params:
            status = "⚠️" if run_time > runtime_threshold else "✓"
            print(f"{status} {len(run_pareto)} solutions, {run_time:.2f}s")

        complete_free_pop(P)
        cleanup_globals()

    total_time = time.time() - total_start_time
    avg_time_per_run = total_time / num_runs if num_runs > 0 else 0.0

    if print_params:
        print(f" Enhanced Complete{agent_info}: {len(all_pareto_solutions)} total solutions, {total_time:.2f}s total, {avg_time_per_run:.2f}s avg/run")

    return {
        'pareto_solutions': np.array(all_pareto_solutions) if all_pareto_solutions else np.array([]),
        'total_time': total_time,
        'avg_time_per_run': avg_time_per_run,
        'run_times': run_times,
        'parameters': {
            'alpha': alpha,
            'kappa': kappa,
            'L': L,
            'small_value': smallValue
        },
        'num_solutions': len(all_pareto_solutions),
        'num_runs': num_runs,
        'num_iterations': num_iterations,
        'operator_strategy': operator_strategy,
        'max_obj1': max_obj1,
        'min_obj1': min_obj1,
        'spread_obj1': spread_obj1,
        'max_obj2': max_obj2,
        'min_obj2': min_obj2,
        'spread_obj2': spread_obj2,
        'agent_id': agent_id,
        'generation': generation
    }

print(" Enhanced MOACP Implementation with All 8 Operators Ready!")

In [89]:
class MindEvolutionOptimizer:
    """Enhanced optimizer with adaptive operator management and knowledge transfer"""
    
    def __init__(self, llm_interface, config_manager, reference_hvs):
        self.llm_interface = llm_interface
        self.config_manager = config_manager
        self.reference_hvs = reference_hvs
        self.optimization_history = []
        self.best_results = {}
        self.agent_population = []
        self.generation = 0
        self.performance_tracker = {}
        self.operator_manager = AdaptiveOperatorManager()
        
    def optimize_instance_with_mind_evolution(self, instance_file, weights_file, nbitems, num_objectives, 
                                           max_generations=10):
        """
        Optimize a specific instance using Mind Evolution with adaptive operators
        """
        # Determine instance signature
        instance_sig = f"{nbitems}_{num_objectives}"
        
        # Get reference data for this instance
        if instance_sig not in self.reference_hvs:
            print(f"❌ No reference data found for instance {instance_sig}")
            return None, None, False
        
        reference_data = self.reference_hvs[instance_sig]
        reference_hv = reference_data['hypervolume']
        target_hv = reference_hv * 1.05  # 5% improvement target
        
        print(f"\n🧠 ENHANCED MIND EVOLUTION OPTIMIZATION")
        print(f"Instance: {instance_file}")
        print(f"Items: {nbitems}, Objectives: {num_objectives}")
        print(f"Reference HV: {reference_hv:,.0f}")
        print(f"Target HV: {target_hv:,.0f}")
        print(f"Gap to close: {target_hv - reference_hv:,.0f}")
        
        # Extract instance features
        instance_features = self.config_manager.extract_instance_features(
            instance_file, weights_file, nbitems, num_objectives
        )
        
        print(f"Instance signature: {instance_features['signature']}")
        print(f"Instance size: {instance_features['instance_size']}")
        print(f"Subclass: {instance_features['subclass']}")
        
        # Show optimal operators for this instance
        optimal_operators = self.operator_manager.get_optimal_operators(instance_features, 0)
        print(f"Optimal operators for generation 0: {optimal_operators}")
        
        # Try to transfer knowledge from similar instances
        transferred_config = self.config_manager.transfer_knowledge(instance_features['signature'], 0)
        if transferred_config:
            print(f"✅ Transferred knowledge from similar instance")
            print(f"   Reasoning: {transferred_config['reasoning']}")
        
        # Initialize agent population
        print(f"\n🔬 Initializing agent population...")
        self.agent_population = self.llm_interface.initialize_evolutionary_population(
            instance_features, target_hv, 0
        )
        
        # Add transferred config if available
        if transferred_config:
            # Replace the worst config with transferred one
            self.agent_population[-1] = transferred_config
            print(f"✅ Added transferred config to population")
        
        print(f"Initialized {len(self.agent_population)} agents")
        
        best_result = None
        best_hv = 0
        best_config = None
        dominance_achieved = False
        performance_history = []
        
        for generation in range(max_generations):
            self.generation = generation
            print(f"\n🧬 Generation {generation + 1}/{max_generations}")
            
            # Show current optimal operators
            current_optimal = self.operator_manager.get_optimal_operators(instance_features, generation)
            print(f"  Optimal operators: {current_optimal}")
            
            # Evaluate all agents
            agent_results = []
            for i, config in enumerate(self.agent_population):
                print(f"  🤖 Agent {i+1}/{len(self.agent_population)}: α={config['alpha']}, κ={config['kappa']:.3f}, L={config['L']}")
                print(f"      Operators: {config['operator_strategy']}")
                
                # Run with current agent configuration
                result = run_moacp_mind_evolution(
                    instance_file=instance_file,
                    weights_file=weights_file,
                    nbitems=nbitems,
                    num_objectives=num_objectives,
                    custom_params=config,
                    operator_strategy=config['operator_strategy'],
                    print_params=False,
                    runtime_threshold=config['runtime_threshold'],
                    num_runs=6,  # Reduced runs for faster evolution
                    num_iterations=80,  # Reduced iterations for faster evolution
                    agent_id=i,
                    generation=generation
                )
                
                # Calculate hypervolume
                hv = calculate_hypervolume_2d(result['pareto_solutions'])
                gap_to_target = target_hv - hv
                improvement = ((hv - reference_hv) / reference_hv) * 100
                
                print(f"    HV: {hv:,.0f} ({improvement:.2f}% improvement)")
                
                # Add performance to config for future reference
                config_with_perf = config.copy()
                config_with_perf['performance'] = hv
                
                agent_results.append({
                    'agent_id': i,
                    'config': config_with_perf,
                    'result': result,
                    'hv': hv,
                    'improvement': improvement,
                    'gap': gap_to_target
                })
                
                # Update operator performance
                for operator in config['operator_strategy']:
                    self.operator_manager.update_operator_performance(operator, improvement, result['total_time'])
                
                # Update best result
                if hv > best_hv:
                    best_hv = hv
                    best_result = result
                    best_config = config.copy()
                    print(f"    ✨ New best! HV: {hv:,.0f} ({improvement:.2f}% improvement)")
                
                # Check for dominance
                if hv >= target_hv:
                    excess = hv - target_hv
                    print(f"    🎉 DOMINANCE ACHIEVED! Excess={excess:,.0f}")
                    dominance_achieved = True
                    break
            
            if dominance_achieved:
                break
            
            # Sort agents by performance
            agent_results.sort(key=lambda x: x['hv'], reverse=True)
            
            # Print generation summary
            print(f"  📊 Generation {generation + 1} Summary:")
            print(f"    Best HV: {agent_results[0]['hv']:,.0f} ({agent_results[0]['improvement']:.2f}% improvement)")
            print(f"    Worst HV: {agent_results[-1]['hv']:,.0f} ({agent_results[-1]['improvement']:.2f}% improvement)")
            print(f"    Average HV: {np.mean([r['hv'] for r in agent_results]):,.0f}")
            
            # Track performance history
            generation_performance = {
                'generation': generation,
                'best_hv': agent_results[0]['hv'],
                'avg_hv': np.mean([r['hv'] for r in agent_results]),
                'worst_hv': agent_results[-1]['hv'],
                'improvement': agent_results[0]['improvement']
            }
            performance_history.append(generation_performance)
            
            # Show operator statistics
            if generation > 0:
                op_stats = self.operator_manager.get_operator_statistics()
                print(f"  🔧 Operator Performance:")
                for op, stats in op_stats.items():
                    if stats['usage_count'] > 0:
                        print(f"    {op}: success={stats['success_rate']:.2f}, usage={stats['usage_count']}")
            
            # Adaptive parameter adjustment based on performance trend
            if generation > 0:
                recent_improvements = [h['improvement'] for h in performance_history[-3:]]
                avg_improvement = np.mean(recent_improvements)
                
                if avg_improvement < 0.5:  # Less than 0.5% improvement
                    print(f"    ⚠️ Low improvement detected ({avg_improvement:.2f}%), adjusting strategy")
                    # Increase exploration in next generation
                    for i, agent_result in enumerate(agent_results[:3]):  # Adjust top 3 agents
                        if i < len(self.agent_population):
                            adjusted_config = self._adaptive_parameter_adjustment(
                                agent_result['config'], performance_history, generation
                            )
                            self.agent_population[i] = adjusted_config
            
            # Create next generation
            if generation < max_generations - 1:
                print(f"  🔄 Creating next generation...")
                next_generation = []
                
                # Elitism: keep top 2 agents
                next_generation.append(agent_results[0]['config'].copy())
                next_generation.append(agent_results[1]['config'].copy())
                
                # Crossover: create 2 offspring from top parents
                parent1 = agent_results[0]['config']
                parent2 = agent_results[1]['config']
                
                offspring1 = self.llm_interface.evolutionary_crossover(
                    parent1, parent2, instance_features, generation
                )
                offspring2 = self.llm_interface.evolutionary_crossover(
                    parent2, parent1, instance_features, generation
                )
                
                next_generation.append(offspring1)
                next_generation.append(offspring2)
                
                # Mutation: mutate 1 agent
                mutation_target = agent_results[2]['config']  # Third best
                
                # Adaptive mutation strength based on performance
                if generation_performance['improvement'] < 0.5:
                    mutation_strength = 'high'  # More exploration needed
                elif generation_performance['improvement'] > 1.5:
                    mutation_strength = 'low'  # Fine-tuning
                else:
                    mutation_strength = 'medium'
                
                mutated = self.llm_interface.evolutionary_mutation(
                    mutation_target, instance_features, mutation_strength, generation
                )
                
                next_generation.append(mutated)
                
                self.agent_population = next_generation
                print(f"    Created {len(next_generation)} agents for next generation")
        
        # Update best configuration for this instance type
        if best_config:
            self.config_manager.update_best_config(
                instance_features['subclass'], best_config, best_hv, best_result['total_time']
            )
            
            # Update LLM knowledge base
            self.llm_interface.update_knowledge_base(
                instance_features['subclass'], 
                [agent['config'] for agent in agent_results[:5]],  # Top 5 configs
                [agent['hv'] for agent in agent_results[:5]]
            )
        
        # Store best result
        if instance_features['signature'] not in self.best_results:
            self.best_results[instance_features['signature']] = {}
        
        self.best_results[instance_features['signature']] = {
            'result': best_result,
            'config': best_config,
            'hv': best_hv,
            'dominance_achieved': dominance_achieved,
            'reference_hv': reference_hv,
            'target_hv': target_hv,
            'generations': generation + 1,
            'performance_history': performance_history
        }
        
        # Show final operator statistics
        print(f"\n📈 Final Operator Statistics:")
        op_stats = self.operator_manager.get_operator_statistics()
        for op, stats in sorted(op_stats.items(), key=lambda x: x[1]['success_rate'], reverse=True):
            if stats['usage_count'] > 0:
                print(f"  {op}: success={stats['success_rate']:.2f}, avg_improvement={stats['avg_improvement']:.2f}%, usage={stats['usage_count']}")
        
        if not dominance_achieved:
            gap_remaining = target_hv - best_hv
            print(f"\n⚠️ Best effort completed. Final HV: {best_hv:,.0f}")
            print(f"Gap remaining: {gap_remaining:,.0f}")
        
        return best_result, best_config, dominance_achieved
    
    def _adaptive_parameter_adjustment(self, config, performance_history, generation):
        """Adjust parameters based on performance history"""
        if len(performance_history) < 2:
            return config
        
        # Calculate recent improvement trend
        recent_improvements = [h['improvement'] for h in performance_history[-3:]]
        avg_improvement = np.mean(recent_improvements)
        
        # If improvement is stagnating, adjust parameters
        if avg_improvement < 0.5:  # Less than 0.5% improvement
            # Increase exploration
            adjusted_config = config.copy()
            
            # Increase population size for more diversity
            adjusted_config['alpha'] = min(60, config['alpha'] + 5)
            
            # Increase local search intensity
            adjusted_config['L'] = min(12, config['L'] + 1)
            
            # Adjust selection pressure
            adjusted_config['kappa'] = max(0.03, config['kappa'] - 0.02)
            
            # Add more diverse operators
            if 'mutation' not in adjusted_config['operator_strategy']:
                adjusted_config['operator_strategy'].append('mutation')
            if 'diversify' not in adjusted_config['operator_strategy']:
                adjusted_config['operator_strategy'].append('diversify')
            
            adjusted_config['reasoning'] = f"Adjusted for stagnation at generation {generation}"
            return adjusted_config
        
        # If improvement is good, fine-tune
        elif avg_improvement > 1.5:  # More than 1.5% improvement
            adjusted_config = config.copy()
            
            # Fine-tune parameters
            adjusted_config['alpha'] = config['alpha'] + np.random.choice([-2, 0, 2])
            adjusted_config['kappa'] = config['kappa'] + np.random.choice([-0.01, 0, 0.01])
            adjusted_config['L'] = config['L'] + np.random.choice([-1, 0, 1])
            
            # Ensure parameters stay in bounds
            adjusted_config['alpha'] = min(60, max(20, adjusted_config['alpha']))
            adjusted_config['kappa'] = min(0.25, max(0.03, adjusted_config['kappa']))
            adjusted_config['L'] = min(12, max(2, adjusted_config['L']))
            
            adjusted_config['reasoning'] = f"Fine-tuned for good improvement at generation {generation}"
            return adjusted_config
        
        # Otherwise, keep current config
        return config

print("\n=== ENHANCED MIND EVOLUTION OPTIMIZER READY ===")


=== ENHANCED MIND EVOLUTION OPTIMIZER READY ===


In [90]:
def calculate_hypervolume_2d(pareto_points, reference_point=None):
    """Calculate hypervolume for 2D Pareto front (maximization)"""
    if len(pareto_points) == 0:
        return 0.0
    
    # Sort points by first objective (ascending for maximization)
    sorted_points = pareto_points[np.argsort(pareto_points[:, 0])]
    
    # Set reference point if not provided
    if reference_point is None:
        # Use origin as reference point for maximization
        reference_point = np.zeros(2)
    
    # Calculate hypervolume
    hypervolume = 0.0
    prev_x = reference_point[0]
    
    for point in sorted_points:
        # For maximization, we calculate area above the point
        width = point[0] - prev_x
        height = point[1] - reference_point[1]
        hypervolume += width * height
        prev_x = point[0]
    
    return hypervolume

def calculate_reference_hv_for_instance(result_file):
    """Calculate reference hypervolume for a specific instance result file"""
    try:
        # Load the results
        data = np.loadtxt(result_file)
        if data.ndim == 1:
            data = data.reshape(1, -1)
        
        # Remove duplicates
        data_unique = np.unique(data, axis=0)
        
        # Extract Pareto front (maximization)
        def is_pareto_efficient(points):
            is_efficient = np.ones(points.shape[0], dtype=bool)
            for i, c in enumerate(points):
                if is_efficient[i]:
                    is_efficient[is_efficient] = np.any(points[is_efficient]>=c, axis=1)
                    is_efficient[i] = True
            return is_efficient
        
        pareto_mask = is_pareto_efficient(data_unique)
        pareto_points = data_unique[pareto_mask]
        
        # Calculate hypervolume
        hv = calculate_hypervolume_2d(pareto_points)
        
        return {
            'all_solutions': data,
            'unique_solutions': data_unique,
            'pareto_solutions': pareto_points,
            'num_pareto': len(pareto_points),
            'hypervolume': hv
        }
    except Exception as e:
        print(f"Error loading result file {result_file}: {e}")
        return None

def visualize_pareto_comparison(reference_file, optimized_file, title="Pareto Front Comparison"):
    """Visualize comparison between reference and optimized Pareto fronts"""
    # Load reference data
    ref_data = calculate_reference_hv_for_instance(reference_file)
    if not ref_data:
        print(f"Error loading reference file {reference_file}")
        return
    
    # Load optimized data
    opt_data = calculate_reference_hv_for_instance(optimized_file)
    if not opt_data:
        print(f"Error loading optimized file {optimized_file}")
        return
    
    # Create plot
    plt.figure(figsize=(12, 8))
    
    # Plot reference Pareto front
    ref_pareto = ref_data['pareto_solutions']
    plt.scatter(ref_pareto[:, 0], ref_pareto[:, 1], 
                c='blue', alpha=0.7, label=f'Reference (HV: {ref_data["hypervolume"]:,.0f})')
    
    # Plot optimized Pareto front
    opt_pareto = opt_data['pareto_solutions']
    plt.scatter(opt_pareto[:, 0], opt_pareto[:, 1], 
                c='red', alpha=0.7, label=f'Optimized (HV: {opt_data["hypervolume"]:,.0f})')
    
    # Calculate improvement
    improvement = ((opt_data['hypervolume'] - ref_data['hypervolume']) / ref_data['hypervolume']) * 100
    
    plt.title(f"{title}\nImprovement: {improvement:.2f}%")
    plt.xlabel('Objective 1')
    plt.ylabel('Objective 2')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    
    # Save plot
    filename = f"pareto_comparison_{os.path.basename(reference_file).replace('.txt', '')}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✅ Comparison plot saved: {filename}")
    plt.show()
    
    return improvement

def analyze_performance_across_instances(results_summary):
    """Analyze performance across all instances"""
    instances = list(results_summary.keys())
    improvements = [results_summary[inst]['improvement'] for inst in instances]
    dominance_rates = [1 if results_summary[inst]['dominance_achieved'] else 0 for inst in instances]
    
    # Create performance summary
    avg_improvement = np.mean(improvements)
    dominance_rate = np.mean(dominance_rates) * 100
    
    print(f"\n📊 PERFORMANCE ANALYSIS")
    print(f"Average improvement: {avg_improvement:.2f}%")
    print(f"Dominance rate: {dominance_rate:.1f}%")
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Improvement bar chart
    ax1.bar(instances, improvements, color='skyblue')
    ax1.set_xlabel('Instance')
    ax1.set_ylabel('Improvement (%)')
    ax1.set_title('Improvement by Instance')
    ax1.grid(True, axis='y')
    
    # Add average line
    ax1.axhline(y=avg_improvement, color='red', linestyle='--', label=f'Average: {avg_improvement:.2f}%')
    ax1.legend()
    
    # Dominance rate pie chart
    dominance_count = sum(dominance_rates)
    non_dominance_count = len(dominance_rates) - dominance_count
    
    ax2.pie([dominance_count, non_dominance_count], 
            labels=['Dominance Achieved', 'No Dominance'],
            colors=['lightgreen', 'lightcoral'],
            autopct='%1.1f%%',
            startangle=90)
    ax2.set_title(f'Dominance Rate: {dominance_rate:.1f}%')
    
    plt.tight_layout()
    plt.savefig('performance_analysis.png', dpi=300, bbox_inches='tight')
    print("✅ Performance analysis saved: performance_analysis.png")
    plt.show()
    
    return {
        'avg_improvement': avg_improvement,
        'dominance_rate': dominance_rate,
        'instances': instances,
        'improvements': improvements
    }

print("✅ Utility functions ready!")

✅ Utility functions ready!


In [91]:
# Calculate reference hypervolumes if not already done
reference_hvs = {}
instance_files = [
    ("250_2", "2502_Resulats.txt"),
    ("500_2", "5002_Resulats.txt"),
    ("750_2", "7502_Resulats.txt")
]

print("Calculating reference hypervolumes for all instances...")
for instance_sig, result_file in instance_files:
    print(f"Processing {result_file}...")
    ref_data = calculate_reference_hv_for_instance(result_file)
    if ref_data:
        reference_hvs[instance_sig] = ref_data
        print(f"✓ {instance_sig}: HV = {ref_data['hypervolume']:,.0f}, Pareto points = {ref_data['num_pareto']}")
    else:
        print(f"❌ Failed to process {result_file}")

print("\nReference hypervolumes calculated successfully!")

# Initialize improved LLM interface
try:
    mind_evolution_llm = MindEvolutionLLMInterface(model_path="llama3:latest", temperature=0.7)
    
    if mind_evolution_llm.connection_status == "ollama_connected":
        print("✅ Enhanced Mind Evolution LLaMA-3 interface working!")
        llm_working = True
    else:
        print("⚠️ Enhanced Mind Evolution LLaMA-3 interface not working, using fallback")
        llm_working = False
        
except Exception as e:
    print(f"❌ Failed to initialize Enhanced Mind Evolution LLaMA-3: {e}")
    llm_working = False
    mind_evolution_llm = None

# Initialize Enhanced Mind Evolution optimizer
mind_evolution_optimizer = MindEvolutionOptimizer(mind_evolution_llm, instance_config_manager, reference_hvs)
print("✅ Enhanced Mind Evolution Optimizer Ready!")

# Optimize all instances
instances_to_optimize = [
    ("./multiobjectives/250.2.txt", "./multiobjectives/Weights_2obj_FQ200.txt", 250, 2),
    ("./multiobjectives/500.2.txt", "./multiobjectives/Weights_2obj_FQ200.txt", 500, 2),
    ("./multiobjectives/750.2.txt", "./multiobjectives/Weights_2obj_FQ200.txt", 750, 2)
]

results_summary = {}

for instance_file, weights_file, nbitems, num_objectives in instances_to_optimize:
    print(f"\n{'='*60}")
    print(f"ENHANCED MIND EVOLUTION OPTIMIZATION: {instance_file}")
    print(f"{'='*60}")
    
    # Check if files exist
    if not os.path.exists(instance_file):
        print(f"❌ Instance file not found: {instance_file}")
        continue
    
    if not os.path.exists(weights_file):
        print(f"❌ Weights file not found: {weights_file}")
        continue
    
    # Optimize this instance
    result, config, dominance_achieved = mind_evolution_optimizer.optimize_instance_with_mind_evolution(
        instance_file=instance_file,
        weights_file=weights_file,
        nbitems=nbitems,
        num_objectives=num_objectives,
        max_generations=8  # Reduced for faster execution
    )
    
    if result is None:
        print(f"❌ Failed to optimize {instance_file}")
        continue
    
    # Get instance signature
    instance_sig = f"{nbitems}_{num_objectives}"
    
    # Get reference data
    reference_data = reference_hvs[instance_sig]
    reference_hv = reference_data['hypervolume']
    
    # Calculate final hypervolume
    final_hv = calculate_hypervolume_2d(result['pareto_solutions'])
    improvement = ((final_hv - reference_hv) / reference_hv) * 100
    
    # Store results
    results_summary[instance_sig] = {
        'result': result,
        'config': config,
        'hv': final_hv,
        'reference_hv': reference_hv,
        'improvement': improvement,
        'dominance_achieved': dominance_achieved
    }
    
    # Save results
    output_file = f"enhanced_mind_evolution_optimized_{instance_sig}.txt"
    np.savetxt(output_file, result['pareto_solutions'])
    print(f"Results saved to {output_file}")
    
    # Visualize comparison
    reference_file = f"{instance_sig.replace('_', '')}_Resulats.txt"
    if os.path.exists(reference_file):
        visualize_pareto_comparison(reference_file, output_file, f"Enhanced Pareto Front Comparison - {instance_sig}")
    
    print(f"\n🏆 ENHANCED MIND EVOLUTION OPTIMIZATION COMPLETE")
    print(f"Final HV: {final_hv:,.0f}")
    print(f"Reference HV: {reference_hv:,.0f}")
    print(f"Improvement: {improvement:.2f}%")
    print(f"Dominance Achieved: {'YES' if dominance_achieved else 'NO'}")

# Print overall summary
print(f"\n{'='*60}")
print("OVERALL ENHANCED OPTIMIZATION SUMMARY")
print(f"{'='*60}")

for instance_sig, results in results_summary.items():
    print(f"\n{instance_sig}:")
    print(f"  Final HV: {results['hv']:,.0f}")
    print(f"  Reference HV: {results['reference_hv']:,.0f}")
    print(f"  Improvement: {results['improvement']:.2f}%")
    print(f"  Dominance Achieved: {'YES' if results['dominance_achieved'] else 'NO'}")
    if results['config']:
        print(f"  Best Config: α={results['config']['alpha']}, κ={results['config']['kappa']:.3f}, L={results['config']['L']}")
        print(f"  Operators: {results['config']['operator_strategy']}")

# Analyze performance across instances
if results_summary:
    performance_analysis = analyze_performance_across_instances(results_summary)

Calculating reference hypervolumes for all instances...
Processing 2502_Resulats.txt...
✓ 250_2: HV = 97,640,877, Pareto points = 236
Processing 5002_Resulats.txt...
✓ 500_2: HV = 405,521,824, Pareto points = 396
Processing 7502_Resulats.txt...
✓ 750_2: HV = 898,857,593, Pareto points = 412

Reference hypervolumes calculated successfully!
✅ Ollama connected with llama3:latest available
✅ Enhanced Mind Evolution LLaMA-3 interface working!
✅ Enhanced Mind Evolution Optimizer Ready!

ENHANCED MIND EVOLUTION OPTIMIZATION: ./multiobjectives/250.2.txt

🧠 ENHANCED MIND EVOLUTION OPTIMIZATION
Instance: ./multiobjectives/250.2.txt
Items: 250, Objectives: 2
Reference HV: 97,640,877
Target HV: 102,522,921
Gap to close: 4,882,044
Instance signature: 250_2
Instance size: small
Subclass: very_tight_capacity_2obj_low_corr_high_skew
Optimal operators for generation 0: ['swap', 'repair', 'mutation', 'diversify']

🔬 Initializing agent population...
✅ Successfully generated exploration-focused config fro


KeyboardInterrupt

